# From Dataset Integrity to Explainable Deep Learning for Retinal OCT Classification

## Overview

Optical Coherence Tomography (OCT) provides high-resolution cross-sectional
images of retinal structure and is widely used in ophthalmic imaging.

This project investigates deep-learning approaches for classifying retinal
OCT images into four categories:

- **CNV** — Choroidal Neovascularization
- **DME** — Diabetic Macular Edema
- **DRUSEN** — Drusen
- **NORMAL** — Normal retina

Rather than treating classification as only a model-training problem, the
project follows a progressive experimental workflow covering:

- dataset integrity and leakage assessment,
- patient-disjoint evaluation,
- class imbalance,
- image preprocessing,
- compact CNN baselines,
- transfer learning,
- explainability with Grad-CAM,
- confidence calibration,
- patient-level failure analysis,
- and independent test evaluation.

The goal is not only to obtain strong classification performance, but also
to examine whether the experimental design is reliable and whether model
performance generalizes to an independent patient population.

---

## Research Questions

1. How effectively can a compact CNN classify retinal OCT images?
2. Does class-weighted training improve minority-class recognition?
3. Does preserving OCT image geometry improve classification performance?
4. How does a compact CNN compare with a pretrained ResNet18?
5. Which image regions influence model predictions?
6. Are prediction errors concentrated among particular patients or classes?
7. Does strong validation performance translate to the independent test set?

---

## Experimental Workflow

Dataset Integrity Audit  
↓  
Leakage Assessment  
↓  
Leakage-Safer Dataset Selection  
↓  
Patient-Disjoint Validation Split  
↓  
Custom CNN Baseline  
↓  
Class-Imbalance Experiment  
↓  
Aspect-Preserving Preprocessing  
↓  
Transfer Learning with ResNet18  
↓  
Grad-CAM and Error Analysis  
↓  
Confidence Calibration  
↓  
Independent Test Evaluation  
↓  
Distribution-Shift Analysis

# 1. Dataset Integrity Audit

The project initially began with the commonly distributed OCT2017
Version 2 dataset.

Before training a neural network, the dataset was examined for:

- class distribution,
- image dimensions,
- patient identifiers,
- repeated filenames,
- and overlap between the supplied training, validation, and test folders.

This step was performed because image-level accuracy can be substantially
overestimated when scans from the same subject appear in multiple
experimental partitions.

In [ ]:
# 1. Imports and Reproducibility
import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f'Random seed: {SEED}')
print('Basic libraries imported successfully.')


In [ ]:
# 2. Locate OCT2017 Version 2 Splits
INPUT_ROOT = Path('/kaggle/input')
CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
candidate_split_sets = []
for root, dirs, files in os.walk(INPUT_ROOT):
    current_path = Path(root)
    if '__MACOSX' in current_path.parts:
        dirs[:] = []
        continue
    dirs[:] = [d for d in dirs if d != '__MACOSX' and (not d.startswith('.'))]
    directory_lookup = {d.lower(): d for d in dirs}
    if 'train' not in directory_lookup or 'test' not in directory_lookup or ('val' not in directory_lookup and 'validation' not in directory_lookup):
        continue
    val_name = directory_lookup['val'] if 'val' in directory_lookup else directory_lookup['validation']
    proposed_splits = {'train': current_path / directory_lookup['train'], 'test': current_path / directory_lookup['test'], 'val': current_path / val_name}
    valid_structure = True
    for split_path in proposed_splits.values():
        class_folders = {folder.name.upper() for folder in split_path.iterdir() if folder.is_dir() and (not folder.name.startswith('.')) and (folder.name != '__MACOSX')}
        if not set(CLASSES).issubset(class_folders):
            valid_structure = False
            break
    if valid_structure:
        candidate_split_sets.append(proposed_splits)
if len(candidate_split_sets) == 0:
    raise RuntimeError('Could not locate OCT2017 V2 train/test/validation structure.')
if len(candidate_split_sets) > 1:
    print('Multiple candidate V2 datasets detected:')
    for candidate in candidate_split_sets:
        print(candidate['train'].parent)
    raise RuntimeError('Multiple V2 candidates detected. Select the intended dataset explicitly.')
split_dirs = candidate_split_sets[0]
print('Detected OCT2017 V2 splits:\n')
for split_name, split_path in split_dirs.items():
    print(f'{split_name:10s} -> {split_path}')


### Image Counts


In [ ]:
# 3. Dataset Class Distribution
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

def get_image_files(folder):
    """
    Return real image files while excluding macOS metadata
    and hidden files.
    """
    return [file for file in folder.iterdir() if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS and (not file.name.startswith('._')) and (not file.name.startswith('.'))]
records = []
for split_name, split_path in split_dirs.items():
    for class_name in CLASSES:
        class_path = split_path / class_name
        image_files = get_image_files(class_path)
        records.append({'split': split_name, 'class': class_name, 'num_images': len(image_files)})
distribution_df = pd.DataFrame(records)
print(distribution_df)


In [ ]:
pivot_table = distribution_df.pivot(index='class', columns='split', values='num_images')
pivot_table


### Class Distribution


In [ ]:
# 4. Visualize Class Distribution
ax = distribution_df.pivot(index='class', columns='split', values='num_images').plot(kind='bar', figsize=(10, 5))
plt.title('OCT Dataset Class Distribution')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Representative OCT Images


In [ ]:
# 5. Representative Images
train_dir = split_dirs.get('train')
if train_dir is None:
    raise ValueError('Training directory was not detected.')
fig, axes = plt.subplots(nrows=len(CLASSES), ncols=4, figsize=(12, 10))
for row, class_name in enumerate(CLASSES):
    class_path = train_dir / class_name
    image_files = get_image_files(class_path)
    selected_images = random.sample(image_files, min(4, len(image_files)))
    for col, image_path in enumerate(selected_images):
        with Image.open(image_path) as image:
            image = image.convert('L')
            axes[row, col].imshow(image, cmap='gray')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(class_name, fontsize=12, fontweight='bold')
plt.suptitle('Representative Retinal OCT Images', fontsize=15)
plt.tight_layout()
plt.show()


### Image Dimension Analysis


In [ ]:
# 6. Image Dimension Analysis
image_info = []
SAMPLES_PER_CLASS = 200
for class_name in CLASSES:
    class_path = train_dir / class_name
    files = get_image_files(class_path)
    selected_files = random.sample(files, min(SAMPLES_PER_CLASS, len(files)))
    for image_path in selected_files:
        with Image.open(image_path) as img:
            image_info.append({'class': class_name, 'width': img.width, 'height': img.height, 'mode': img.mode})
image_info_df = pd.DataFrame(image_info)
print(image_info_df.head())
print('\nImage width statistics:')
print(image_info_df['width'].describe())
print('\nImage height statistics:')
print(image_info_df['height'].describe())
print('\nImage modes:')
print(image_info_df['mode'].value_counts())


### Aspect-Ratio Analysis


In [ ]:
image_info_df['aspect_ratio'] = image_info_df['width'] / image_info_df['height']
print(image_info_df.groupby('class')[['width', 'height', 'aspect_ratio']].mean().round(2))


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(image_info_df['aspect_ratio'], bins=30)
plt.title('Distribution of OCT Image Aspect Ratios')
plt.xlabel('Width / Height')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


### Patient-ID Parsing


In [ ]:
# 7. Inspect File Naming Structure
for class_name in CLASSES:
    files = get_image_files(train_dir / class_name)
    print(f'\n{class_name}:')
    for file in files[:5]:
        print('   ', file.name)


In [ ]:
def extract_patient_id(image_path):
    """
    Extract randomized patient ID from Kermany OCT filename.

    Example:
        CNV-8086850-26.jpeg
            ->
        8086850
    """
    parts = image_path.stem.split('-')
    if len(parts) >= 3:
        return parts[1]
    return image_path.stem


In [ ]:
example_path = Path('/kaggle/input/example/CNV-8086850-26.jpeg')
print('Filename:', example_path.name)
print('Extracted patient ID:', extract_patient_id(example_path))


### Unique Patient Counts


In [ ]:
# 8. Patient-Level Dataset Analysis
patient_records = []
for split_name, split_path in split_dirs.items():
    for class_name in CLASSES:
        class_path = split_path / class_name
        files = get_image_files(class_path)
        patient_ids = {extract_patient_id(file) for file in files}
        patient_records.append({'split': split_name, 'class': class_name, 'images': len(files), 'unique_patients': len(patient_ids)})
patient_df = pd.DataFrame(patient_records)
patient_df


### Dataset Metadata


In [ ]:
# 10. Build Complete Dataset Metadata
all_records = []
for source_split, split_path in split_dirs.items():
    for class_name in CLASSES:
        class_path = split_path / class_name
        image_files = get_image_files(class_path)
        for image_path in image_files:
            all_records.append({'source_split': source_split, 'class': class_name, 'patient_id': extract_patient_id(image_path), 'filename': image_path.name, 'path': str(image_path)})
metadata_df = pd.DataFrame(all_records)
print('Total image records:', len(metadata_df))
print('Unique filenames:', metadata_df['filename'].nunique())
print('Unique patient IDs:', metadata_df['patient_id'].nunique())
print('\nClass distribution:')
print(metadata_df['class'].value_counts())


### Patient Leakage Analysis


In [ ]:
# 11. Patient Leakage Analysis
patients_by_split = {}
for split_name in metadata_df['source_split'].unique():
    patients_by_split[split_name] = set(metadata_df.loc[metadata_df['source_split'] == split_name, 'patient_id'])
split_names = list(patients_by_split.keys())
for i in range(len(split_names)):
    for j in range(i + 1, len(split_names)):
        split_a = split_names[i]
        split_b = split_names[j]
        overlap = patients_by_split[split_a] & patients_by_split[split_b]
        print(f'{split_a:5s} vs {split_b:5s}: {len(overlap)} overlapping patients')


### Multi-Class Patient IDs


In [ ]:
# 12. Check Whether Patients Appear in Multiple Classes
patient_class_counts = metadata_df.groupby('patient_id')['class'].nunique()
multi_class_patients = patient_class_counts[patient_class_counts > 1]
print('Patients associated with more than one class:', len(multi_class_patients))
if len(multi_class_patients) > 0:
    example_ids = multi_class_patients.index[:10]
    display(metadata_df[metadata_df['patient_id'].isin(example_ids)][['patient_id', 'class', 'source_split', 'filename']].sort_values(['patient_id', 'class']))


### Duplicate Filename Analysis


In [ ]:
# 13. Duplicate Filename Analysis
duplicate_filename_mask = metadata_df.duplicated(subset='filename', keep=False)
duplicate_filename_df = metadata_df[duplicate_filename_mask].sort_values('filename')
print('Rows involved in duplicate filenames:', len(duplicate_filename_df))
print('Number of duplicated filename groups:', duplicate_filename_df['filename'].nunique())
if len(duplicate_filename_df) > 0:
    display(duplicate_filename_df[['filename', 'patient_id', 'class', 'source_split']].head(30))


### Images per Patient


In [ ]:
# 14. Images per Patient
images_per_patient = metadata_df.groupby(['patient_id', 'class']).size().reset_index(name='num_images')
print(images_per_patient.groupby('class')['num_images'].describe().round(2))


In [ ]:
plt.figure(figsize=(9, 5))
data_to_plot = [images_per_patient.loc[images_per_patient['class'] == class_name, 'num_images'].values for class_name in CLASSES]
plt.boxplot(data_to_plot, tick_labels=CLASSES, showfliers=False)
plt.title('Number of OCT Images per Patient')
plt.xlabel('Class')
plt.ylabel('Images per Patient')
plt.tight_layout()
plt.show()


### Duplicate File Locations


In [ ]:
# 15. Investigate Duplicate Filename Locations
duplicate_summary = duplicate_filename_df.groupby('filename')['source_split'].apply(lambda x: ' + '.join(sorted(x))).value_counts()
print('Duplicate filename split combinations:\n')
print(duplicate_summary)


In [ ]:
duplicate_consistency = duplicate_filename_df.groupby('filename').agg(rows=('filename', 'size'), num_splits=('source_split', 'nunique'), num_classes=('class', 'nunique'), num_patient_ids=('patient_id', 'nunique'))
print('Duplicate groups:', len(duplicate_consistency))
print('Groups with conflicting class labels:', (duplicate_consistency['num_classes'] > 1).sum())
print('Groups with conflicting patient IDs:', (duplicate_consistency['num_patient_ids'] > 1).sum())
display(duplicate_consistency.head())


### Duplicate-Content Verification


In [ ]:
# 16. Verify Duplicate Image Contents
import hashlib

def calculate_sha256(file_path):
    """
    Calculate SHA-256 hash of a file.
    """
    sha256 = hashlib.sha256()
    with open(file_path, 'rb') as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            sha256.update(chunk)
    return sha256.hexdigest()


In [ ]:
duplicate_hash_records = []
for _, row in duplicate_filename_df.iterrows():
    duplicate_hash_records.append({'filename': row['filename'], 'source_split': row['source_split'], 'class': row['class'], 'patient_id': row['patient_id'], 'hash': calculate_sha256(row['path'])})
duplicate_hash_df = pd.DataFrame(duplicate_hash_records)


In [ ]:
hash_consistency = duplicate_hash_df.groupby('filename')['hash'].nunique()
identical_groups = (hash_consistency == 1).sum()
different_groups = (hash_consistency > 1).sum()
print('Duplicate filenames with identical image content:', identical_groups)
print('Duplicate filenames with different image content:', different_groups)


### Multi-Class Patient Examination


In [ ]:
# 17. Patient Diagnostic Combination Analysis
patient_class_summary = metadata_df.groupby('patient_id')['class'].apply(lambda x: tuple(sorted(set(x)))).reset_index(name='classes')
patient_class_summary['num_classes'] = patient_class_summary['classes'].apply(len)
multi_class_summary = patient_class_summary[patient_class_summary['num_classes'] > 1]
print('Multi-class patients:', len(multi_class_summary))
display(multi_class_summary.head(20))


In [ ]:
diagnostic_combinations = multi_class_summary['classes'].value_counts()
print('Most common multi-class combinations:\n')
print(diagnostic_combinations.head(15))


In [ ]:
# 18. Pixel-Level Duplicate Analysis
from skimage.metrics import structural_similarity as ssim


In [ ]:
comparison_records = []
duplicate_groups = duplicate_filename_df.groupby('filename')
for filename, group in duplicate_groups:
    if len(group) != 2:
        continue
    rows = list(group.itertuples())
    path_a = rows[0].path
    path_b = rows[1].path
    with Image.open(path_a) as img_a:
        img_a = img_a.convert('L')
        array_a = np.asarray(img_a, dtype=np.float32)
    with Image.open(path_b) as img_b:
        img_b = img_b.convert('L')
        array_b = np.asarray(img_b, dtype=np.float32)
    same_shape = array_a.shape == array_b.shape
    if same_shape:
        exact_pixels = np.array_equal(array_a, array_b)
        mae = np.mean(np.abs(array_a - array_b))
        mse = np.mean((array_a - array_b) ** 2)
        ssim_value = ssim(array_a, array_b, data_range=255)
    else:
        exact_pixels = False
        mae = np.nan
        mse = np.nan
        ssim_value = np.nan
    comparison_records.append({'filename': filename, 'split_a': rows[0].source_split, 'split_b': rows[1].source_split, 'shape_a': array_a.shape, 'shape_b': array_b.shape, 'same_shape': same_shape, 'exact_pixels': exact_pixels, 'mae': mae, 'mse': mse, 'ssim': ssim_value})
duplicate_pixel_df = pd.DataFrame(comparison_records)


In [ ]:
print('Duplicate filename pairs:', len(duplicate_pixel_df))
print('Pairs with identical dimensions:', duplicate_pixel_df['same_shape'].sum())
print('Pairs with different dimensions:', (~duplicate_pixel_df['same_shape']).sum())
print('Pairs with exactly identical decoded pixels:', duplicate_pixel_df['exact_pixels'].sum())


In [ ]:
same_shape_df = duplicate_pixel_df[duplicate_pixel_df['same_shape']]
print('\nMAE statistics:')
print(same_shape_df['mae'].describe())
print('\nSSIM statistics:')
print(same_shape_df['ssim'].describe())


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(same_shape_df['ssim'].dropna(), bins=40)
plt.title('SSIM Between Same-Named Images Across Source Splits')
plt.xlabel('SSIM')
plt.ylabel('Number of Image Pairs')
plt.tight_layout()
plt.show()


In [ ]:
lowest_similarity = same_shape_df.sort_values('ssim').head(12)
display(lowest_similarity[['filename', 'split_a', 'split_b', 'mae', 'ssim']])


In [ ]:
# 19. Visual Inspection of Duplicate Candidates
examples = lowest_similarity.head(4)
fig, axes = plt.subplots(len(examples), 2, figsize=(10, 12))
for row_idx, (_, record) in enumerate(examples.iterrows()):
    filename = record['filename']
    matches = duplicate_filename_df[duplicate_filename_df['filename'] == filename]
    paths = matches['path'].tolist()
    splits = matches['source_split'].tolist()
    for col_idx in range(2):
        with Image.open(paths[col_idx]) as img:
            img = img.convert('L')
            axes[row_idx, col_idx].imshow(img, cmap='gray')
        axes[row_idx, col_idx].set_title(f'{filename}\n{splits[col_idx]}')
        axes[row_idx, col_idx].axis('off')
plt.tight_layout()
plt.show()


## Version 2 Audit Findings

The initial dataset contained:

- **84,484 image records**
- **83,605 unique filenames**
- only **32 images** in the supplied validation partition
- **546 overlapping patient identifiers** between the supplied training
  and test partitions
- **879 filenames repeated across dataset partitions**

Further analysis showed that same-named files were not necessarily
pixel-identical images, making simple filename-based deduplication
inappropriate.

Because subject-level independence is particularly important in medical
image evaluation, the supplied Version 2 partition was not used for model
development.

The experiments were therefore continued using **OCT2017 Version 3**,
followed by a new integrity assessment.

# 2. OCT2017 Version 3

OCT2017 Version 3 was used for all subsequent model-development
experiments.

The dataset contains four retinal OCT categories and provides a separate
official test population.

| Class | Training Images | Official Test Images |
|---|---:|---:|
| CNV | 37,205 | 250 |
| DME | 11,348 | 250 |
| DRUSEN | 8,616 | 250 |
| NORMAL | 51,140 | 250 |
| **Total** | **108,309** | **1,000** |

Independent checks performed in this notebook found:

- **4,785 unique training patient IDs**
- **635 unique test patient IDs**
- **0 overlapping patient IDs between train and test**
- **0 duplicated filenames across train and test**

The official 1,000-image test set was reserved exclusively for final
evaluation.

In [ ]:
# 20. Inspect OCT2017 Version 3 Directory Structure
V3_ROOT = Path('/kaggle/input/datasets/ahmadraza9112/kermany-oct2017-v3')
print('V3 root exists:', V3_ROOT.exists())
print('\nTop-level contents:\n')
for item in sorted(V3_ROOT.iterdir()):
    print(item)


In [ ]:
# 21. Detect V3 Train/Test Splits
CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
v3_split_dirs = {}
for root, dirs, files in os.walk(V3_ROOT):
    current_path = Path(root)
    if '__MACOSX' in current_path.parts:
        dirs[:] = []
        continue
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__MACOSX']
    folder_name = current_path.name.strip().lower()
    if folder_name in {'train', 'training', 'test', 'testing'}:
        available_subfolders = {d.strip().upper() for d in dirs}
        if set(CLASSES).issubset(available_subfolders):
            if folder_name in {'train', 'training'}:
                split_name = 'train'
            else:
                split_name = 'test'
            v3_split_dirs[split_name] = current_path
            dirs[:] = []
print('Detected V3 dataset splits:\n')
for split_name, split_path in v3_split_dirs.items():
    print(f'{split_name:10s} -> {split_path}')


In [ ]:
# 22. V3 Image Counts
v3_records = []
for split_name, split_path in v3_split_dirs.items():
    for class_name in CLASSES:
        class_path = split_path / class_name
        image_files = get_image_files(class_path)
        v3_records.append({'split': split_name, 'class': class_name, 'num_images': len(image_files)})
v3_distribution_df = pd.DataFrame(v3_records)
display(v3_distribution_df)


In [ ]:
v3_pivot = v3_distribution_df.pivot(index='class', columns='split', values='num_images')
display(v3_pivot)
print('\nTotal V3 images:', v3_distribution_df['num_images'].sum())


In [ ]:
# 23. Build V3 Metadata Table
v3_metadata_records = []
for split_name, split_path in v3_split_dirs.items():
    for class_name in CLASSES:
        class_path = split_path / class_name
        image_files = get_image_files(class_path)
        for image_path in image_files:
            v3_metadata_records.append({'source_split': split_name, 'class': class_name, 'patient_id': extract_patient_id(image_path), 'filename': image_path.name, 'path': str(image_path)})
v3_metadata_df = pd.DataFrame(v3_metadata_records)
print('Total images:', len(v3_metadata_df))
print('Unique filenames:', v3_metadata_df['filename'].nunique())
print('Unique patient IDs:', v3_metadata_df['patient_id'].nunique())
print('\nClass distribution:')
print(v3_metadata_df['class'].value_counts())


In [ ]:
# 24. Train/Test Patient Leakage Check
train_patients = set(v3_metadata_df.loc[v3_metadata_df['source_split'] == 'train', 'patient_id'])
test_patients = set(v3_metadata_df.loc[v3_metadata_df['source_split'] == 'test', 'patient_id'])
patient_overlap = train_patients & test_patients
print('Unique train patients:', len(train_patients))
print('Unique test patients:', len(test_patients))
print('Train/test overlapping patient IDs:', len(patient_overlap))


In [ ]:
# 25. V3 Duplicate Filename Check
duplicate_v3_mask = v3_metadata_df.duplicated(subset='filename', keep=False)
duplicate_v3_df = v3_metadata_df[duplicate_v3_mask]
print('Rows involved in duplicate filenames:', len(duplicate_v3_df))
print('Duplicate filename groups:', duplicate_v3_df['filename'].nunique())


In [ ]:
cross_split_duplicates = duplicate_v3_df.groupby('filename')['source_split'].nunique()
print('Duplicate filenames appearing in multiple splits:', (cross_split_duplicates > 1).sum())


In [ ]:
# 26. V3 Image Dimension Analysis
v3_dimension_records = []
SAMPLES_PER_CLASS = 200
train_dir_v3 = v3_split_dirs['train']
for class_name in CLASSES:
    class_path = train_dir_v3 / class_name
    files = get_image_files(class_path)
    selected_files = random.sample(files, min(SAMPLES_PER_CLASS, len(files)))
    for image_path in selected_files:
        with Image.open(image_path) as img:
            v3_dimension_records.append({'class': class_name, 'width': img.width, 'height': img.height, 'mode': img.mode})
v3_dimension_df = pd.DataFrame(v3_dimension_records)
print('Width statistics:\n', v3_dimension_df['width'].describe())
print('\nHeight statistics:\n', v3_dimension_df['height'].describe())
print('\nImage modes:\n', v3_dimension_df['mode'].value_counts())


### Dataset Integrity Decision

Because no train–test patient overlap or filename duplication was detected,
Version 3 was retained as the experimental dataset.

The official test set was not used for architecture selection,
preprocessing decisions, loss-function selection, or hyperparameter
development.

# 3. Patient-Disjoint Validation Strategy

The official test set was preserved for final evaluation, so a separate
validation subset was created from the Version 3 training population.

A critical requirement was that all images belonging to the same patient
remain within a single experimental partition.

Candidate group-aware splits were generated and compared according to
validation size and class-distribution similarity.

The selected split contained:

| Split | Images |
|---|---:|
| Training | 97,348 |
| Validation | 10,961 |
| Official Test | 1,000 |

Final verification confirmed:

- Train ∩ Validation patients = **0**
- Train ∩ Test patients = **0**
- Validation ∩ Test patients = **0**

This fixed split was saved and reused for all experiments.

In [ ]:
# 27. V3 Training Population Analysis
v3_train_pool = v3_metadata_df[v3_metadata_df['source_split'] == 'train'].copy().reset_index(drop=True)
v3_test_final = v3_metadata_df[v3_metadata_df['source_split'] == 'test'].copy().reset_index(drop=True)
print('Training-pool images:', len(v3_train_pool))
print('Training-pool patients:', v3_train_pool['patient_id'].nunique())
print('\nOfficial test images:', len(v3_test_final))
print('Official test patients:', v3_test_final['patient_id'].nunique())
print('\nTraining class distribution:')
train_class_summary = v3_train_pool['class'].value_counts().reindex(CLASSES)
print(train_class_summary)
print('\nTraining class percentages:')
print((train_class_summary / len(v3_train_pool) * 100).round(2))


In [ ]:
# 28. Multi-Class Patient Analysis
v3_patient_classes = v3_train_pool.groupby('patient_id')['class'].nunique()
print('Patients appearing in one class:', (v3_patient_classes == 1).sum())
print('Patients appearing in multiple classes:', (v3_patient_classes > 1).sum())
print('\nMaximum number of classes associated with one patient ID:', v3_patient_classes.max())


In [ ]:
# 29. Patient-Aware Validation Split
from sklearn.model_selection import StratifiedGroupKFold


In [ ]:
N_SPLITS = 10
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
X_dummy = np.zeros(len(v3_train_pool))
y = v3_train_pool['class'].values
groups = v3_train_pool['patient_id'].values
overall_proportions = v3_train_pool['class'].value_counts(normalize=True).reindex(CLASSES)


In [ ]:
candidate_results = []
candidate_indices = {}
for fold_number, (train_idx, val_idx) in enumerate(sgkf.split(X_dummy, y, groups), start=1):
    candidate_train = v3_train_pool.iloc[train_idx]
    candidate_val = v3_train_pool.iloc[val_idx]
    val_proportions = candidate_val['class'].value_counts(normalize=True).reindex(CLASSES, fill_value=0)
    class_error = (val_proportions - overall_proportions).abs().sum()
    validation_fraction = len(candidate_val) / len(v3_train_pool)
    size_error = abs(validation_fraction - 0.1)
    score = class_error + 2.0 * size_error
    train_patients_fold = set(candidate_train['patient_id'])
    val_patients_fold = set(candidate_val['patient_id'])
    patient_overlap_fold = len(train_patients_fold & val_patients_fold)
    candidate_results.append({'fold': fold_number, 'train_images': len(candidate_train), 'val_images': len(candidate_val), 'val_fraction': validation_fraction, 'train_patients': candidate_train['patient_id'].nunique(), 'val_patients': candidate_val['patient_id'].nunique(), 'patient_overlap': patient_overlap_fold, 'class_error': class_error, 'size_error': size_error, 'score': score})
    candidate_indices[fold_number] = (train_idx, val_idx)


In [ ]:
candidate_results_df = pd.DataFrame(candidate_results).sort_values('score').reset_index(drop=True)
display(candidate_results_df.round(4))


In [ ]:
best_fold = int(candidate_results_df.iloc[0]['fold'])
best_train_idx, best_val_idx = candidate_indices[best_fold]
print('Selected validation fold:', best_fold)


In [ ]:
# 30. Construct Final Experimental Dataset
train_final_df = v3_train_pool.iloc[best_train_idx].copy()
val_final_df = v3_train_pool.iloc[best_val_idx].copy()
test_final_df = v3_test_final.copy()
train_final_df['experiment_split'] = 'train'
val_final_df['experiment_split'] = 'val'
test_final_df['experiment_split'] = 'test'
experiment_df = pd.concat([train_final_df, val_final_df, test_final_df], ignore_index=True)
print('Total experimental images:', len(experiment_df))
print('\nSplit counts:')
print(experiment_df['experiment_split'].value_counts())


In [ ]:
# 31. Verify Complete Patient Independence
patients = {}
for split_name in ['train', 'val', 'test']:
    patients[split_name] = set(experiment_df.loc[experiment_df['experiment_split'] == split_name, 'patient_id'])
comparisons = [('train', 'val'), ('train', 'test'), ('val', 'test')]
for split_a, split_b in comparisons:
    overlap = patients[split_a] & patients[split_b]
    print(f'{split_a:5s} vs {split_b:5s}: {len(overlap)} overlapping patients')


In [ ]:
# 32. Final Image-Level Distribution
final_image_distribution = experiment_df.groupby(['class', 'experiment_split']).size().unstack(fill_value=0).reindex(CLASSES)
column_order = [col for col in ['train', 'val', 'test'] if col in final_image_distribution.columns]
final_image_distribution = final_image_distribution[column_order]
display(final_image_distribution)


In [ ]:
validation_percentages = val_final_df['class'].value_counts(normalize=True).reindex(CLASSES) * 100
print('Validation class percentages:\n')
print(validation_percentages.round(2))


In [ ]:
# 33. Patient-Level Distribution
patient_distribution = experiment_df.groupby(['experiment_split', 'class'])['patient_id'].nunique().unstack(fill_value=0)
display(patient_distribution)


In [ ]:
# 34. Save Experimental Split
SPLIT_FILE = '/kaggle/working/oct2017_v3_experiment_split.csv'
experiment_df[['filename', 'patient_id', 'class', 'source_split', 'experiment_split', 'path']].to_csv(SPLIT_FILE, index=False)
print('Saved split file:')
print(SPLIT_FILE)


# 4. Image Preprocessing

Exploratory analysis showed that OCT image heights were relatively
consistent, while image widths varied substantially.

A direct resize to 224 × 224 therefore changes the original image aspect
ratio.

Two preprocessing approaches were investigated:

### Direct Square Resizing

All images were resized directly to 224 × 224. This provided a simple
baseline but introduced geometric distortion.

### Aspect-Preserving Resizing

Images were resized while preserving their original aspect ratio and then
symmetrically padded to produce 224 × 224 inputs.

This approach retained more of the original OCT geometry and was later
evaluated experimentally.

Normalization statistics were estimated using training images only.

In [ ]:
# 35. PyTorch Environment
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
print('PyTorch version:', torch.__version__)
print('Torchvision version:', torchvision.__version__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 36. PyTorch Reproducibility
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f'PyTorch random seed set to {SEED}')


In [ ]:
# 37. Locked Experimental Split
print('Training images:', len(train_final_df))
print('Validation images:', len(val_final_df))
print('Test images:', len(test_final_df))
print('\nTotal:', len(train_final_df) + len(val_final_df) + len(test_final_df))


In [ ]:
# 38. Class Mapping
CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
CLASS_TO_IDX = {class_name: idx for idx, class_name in enumerate(CLASSES)}
IDX_TO_CLASS = {idx: class_name for class_name, idx in CLASS_TO_IDX.items()}
print('Class mapping:')
for class_name, idx in CLASS_TO_IDX.items():
    print(f'{class_name:8s} -> {idx}')


In [ ]:
# 39. Estimate Training Image Statistics
STATS_SAMPLE_SIZE = 2000
stats_sample = train_final_df.sample(n=min(STATS_SAMPLE_SIZE, len(train_final_df)), random_state=SEED)
stats_transform = transforms.Compose([transforms.Resize((224, 224), interpolation=InterpolationMode.BILINEAR, antialias=True), transforms.ToTensor()])
pixel_sum = 0.0
pixel_squared_sum = 0.0
pixel_count = 0
for image_path in stats_sample['path']:
    with Image.open(image_path) as img:
        img = img.convert('L')
        tensor = stats_transform(img)
    pixel_sum += tensor.sum().item()
    pixel_squared_sum += (tensor ** 2).sum().item()
    pixel_count += tensor.numel()
TRAIN_MEAN = pixel_sum / pixel_count
TRAIN_VARIANCE = pixel_squared_sum / pixel_count - TRAIN_MEAN ** 2
TRAIN_STD = TRAIN_VARIANCE ** 0.5
print(f'Estimated training mean: {TRAIN_MEAN:.4f}')
print(f'Estimated training std:  {TRAIN_STD:.4f}')


In [ ]:
# 40. Baseline Image Transforms
IMAGE_SIZE = 224
train_transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True), transforms.RandomHorizontalFlip(p=0.5), transforms.ToTensor(), transforms.Normalize(mean=[TRAIN_MEAN], std=[TRAIN_STD])])
evaluation_transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True), transforms.ToTensor(), transforms.Normalize(mean=[TRAIN_MEAN], std=[TRAIN_STD])])
print('Baseline transforms created.')


In [ ]:
# 41. OCT Dataset Class
class OCTDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = row['path']
        class_name = row['class']
        label = CLASS_TO_IDX[class_name]
        with Image.open(image_path) as image:
            image = image.convert('L')
            if self.transform is not None:
                image = self.transform(image)
        return (image, label)


In [ ]:
# 42. Dataset Objects
train_dataset = OCTDataset(dataframe=train_final_df, transform=train_transform)
val_dataset = OCTDataset(dataframe=val_final_df, transform=evaluation_transform)
test_dataset = OCTDataset(dataframe=test_final_df, transform=evaluation_transform)
print('Train dataset:', len(train_dataset))
print('Validation dataset:', len(val_dataset))
print('Test dataset:', len(test_dataset))


In [ ]:
# 43. Dataset Sanity Check
sample_image, sample_label = train_dataset[0]
print('Image tensor shape:', sample_image.shape)
print('Label index:', sample_label)
print('Label name:', IDX_TO_CLASS[sample_label])
print('Tensor minimum:', sample_image.min().item())
print('Tensor maximum:', sample_image.max().item())


In [ ]:
# 44. DataLoaders
BATCH_SIZE = 32
NUM_WORKERS = 2
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=False)
print('Training batches:', len(train_loader))
print('Validation batches:', len(val_loader))
print('Test batches:', len(test_loader))


In [ ]:
# 45. DataLoader Sanity Check
images, labels = next(iter(train_loader))
print('Batch image shape:', images.shape)
print('Batch label shape:', labels.shape)
print('Image dtype:', images.dtype)
print('Label dtype:', labels.dtype)
print('\nFirst 10 labels:')
print(labels[:10])


In [ ]:
# 46. Visualize Training Batch
def denormalize_image(tensor):
    return (tensor * TRAIN_STD + TRAIN_MEAN).clamp(0, 1)
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
axes = axes.flatten()
for i in range(12):
    image = denormalize_image(images[i])
    label = labels[i].item()
    axes[i].imshow(image.squeeze(0), cmap='gray')
    axes[i].set_title(IDX_TO_CLASS[label])
    axes[i].axis('off')
plt.suptitle('Preprocessed Training Batch', fontsize=15)
plt.tight_layout()
plt.show()


# 5. Experiment E01 — Compact CNN Baseline

The first model establishes a simple and transparent baseline.

### Architecture

The custom CNN uses four convolutional stages followed by global average
pooling and a small classifier.

- Input: 1 × 224 × 224 grayscale OCT image
- Trainable parameters: **389,828**
- Loss: standard cross-entropy
- Optimizer: Adam
- Learning rate: 0.001
- Model selection metric: validation Macro F1

No class weighting was used.

In [ ]:
# 47. GPU Configuration
print('CUDA available:', torch.cuda.is_available())
print('Number of GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))
print('Training device:', DEVICE)


In [ ]:
# 48. Custom CNN Baseline
class OCTBaselineCNN(nn.Module):

    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=2), nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=2), nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=2), nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(output_size=(1, 1)))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(p=0.3), nn.Linear(256, num_classes))

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [ ]:
# 49. Instantiate Baseline Model
baseline_model = OCTBaselineCNN(num_classes=len(CLASSES))
baseline_model = baseline_model.to(DEVICE)

def count_parameters(model):
    total = sum((p.numel() for p in model.parameters()))
    trainable = sum((p.numel() for p in model.parameters() if p.requires_grad))
    return (total, trainable)
total_params, trainable_params = count_parameters(baseline_model)
print(baseline_model)
print(f'\nTotal parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')


In [ ]:
# 51. Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(baseline_model.parameters(), lr=0.001)
print('Loss: CrossEntropyLoss')
print('Optimizer: Adam')
print('Learning rate: 0.001')


In [ ]:
# 52. Evaluation Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def calculate_metrics(true_labels, predicted_labels):
    accuracy = accuracy_score(true_labels, predicted_labels)
    macro_precision = precision_score(true_labels, predicted_labels, average='macro', zero_division=0)
    macro_recall = recall_score(true_labels, predicted_labels, average='macro', zero_division=0)
    macro_f1 = f1_score(true_labels, predicted_labels, average='macro', zero_division=0)
    return {'accuracy': accuracy, 'macro_precision': macro_precision, 'macro_recall': macro_recall, 'macro_f1': macro_f1}


In [ ]:
# 53. Training Function
from tqdm.auto import tqdm
USE_AMP = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    true_labels = []
    predicted_labels = []
    progress_bar = tqdm(loader, desc='Training', leave=False)
    for batch_images, batch_labels in progress_bar:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
            outputs = model(batch_images)
            loss = criterion(outputs, batch_labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * batch_images.size(0)
        predictions = outputs.argmax(dim=1)
        true_labels.extend(batch_labels.detach().cpu().numpy())
        predicted_labels.extend(predictions.detach().cpu().numpy())
        progress_bar.set_postfix(loss=f'{loss.item():.4f}')
    epoch_loss = running_loss / len(loader.dataset)
    metrics = calculate_metrics(true_labels, predicted_labels)
    return (epoch_loss, metrics)


In [ ]:
# 54. Validation Function
def evaluate_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    true_labels = []
    predicted_labels = []
    with torch.no_grad():
        for batch_images, batch_labels in tqdm(loader, desc='Validation', leave=False):
            batch_images = batch_images.to(device, non_blocking=True)
            batch_labels = batch_labels.to(device, non_blocking=True)
            with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                outputs = model(batch_images)
                loss = criterion(outputs, batch_labels)
            running_loss += loss.item() * batch_images.size(0)
            predictions = outputs.argmax(dim=1)
            true_labels.extend(batch_labels.detach().cpu().numpy())
            predicted_labels.extend(predictions.detach().cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    metrics = calculate_metrics(true_labels, predicted_labels)
    return (epoch_loss, metrics, np.array(true_labels), np.array(predicted_labels))


In [ ]:
# 55. Baseline Training Configuration
NUM_EPOCHS = 8
BEST_MODEL_PATH = '/kaggle/working/oct_baseline_best.pt'
history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': [], 'train_macro_f1': [], 'val_macro_f1': []}
best_val_f1 = -1.0
best_epoch = 0


In [ ]:
# 56. Train Custom CNN Baseline
for epoch in range(1, NUM_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{NUM_EPOCHS}')
    print('-' * 50)
    train_loss, train_metrics = train_one_epoch(baseline_model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_metrics, val_true, val_pred = evaluate_model(baseline_model, val_loader, criterion, DEVICE)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_accuracy'].append(train_metrics['accuracy'])
    history['val_accuracy'].append(val_metrics['accuracy'])
    history['train_macro_f1'].append(train_metrics['macro_f1'])
    history['val_macro_f1'].append(val_metrics['macro_f1'])
    print(f'Train Loss: {train_loss:.4f}')
    print(f"Train Accuracy: {train_metrics['accuracy']:.4f}")
    print(f"Train Macro F1: {train_metrics['macro_f1']:.4f}")
    print(f'\nVal Loss: {val_loss:.4f}')
    print(f"Val Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"Val Macro F1: {val_metrics['macro_f1']:.4f}")
    if val_metrics['macro_f1'] > best_val_f1:
        best_val_f1 = val_metrics['macro_f1']
        best_epoch = epoch
        torch.save({'epoch': epoch, 'model_state_dict': baseline_model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'val_macro_f1': best_val_f1, 'train_mean': TRAIN_MEAN, 'train_std': TRAIN_STD, 'class_to_idx': CLASS_TO_IDX}, BEST_MODEL_PATH)
        print('\n✓ Best model saved.')
print('\nTraining complete.')
print('Best epoch:', best_epoch)
print('Best validation Macro F1:', round(best_val_f1, 4))


In [ ]:
# 57. Baseline Learning Curves
epochs = range(1, NUM_EPOCHS + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs, history['train_loss'], marker='o', label='Train')
plt.plot(epochs, history['val_loss'], marker='o', label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('Baseline CNN — Loss')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epochs, history['train_macro_f1'], marker='o', label='Train')
plt.plot(epochs, history['val_macro_f1'], marker='o', label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Macro F1')
plt.title('Baseline CNN — Macro F1')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
e01_history_df = pd.DataFrame({'Epoch': range(1, len(history['train_loss']) + 1), 'Train Loss': history['train_loss'], 'Val Loss': history['val_loss'], 'Train Accuracy': history['train_accuracy'], 'Val Accuracy': history['val_accuracy'], 'Train Macro F1': history['train_macro_f1'], 'Val Macro F1': history['val_macro_f1']})
display(e01_history_df.round(4))
e01_history_df.to_csv('/kaggle/working/e01_training_history.csv', index=False)


In [ ]:
# 58. Stable DataLoaders for Kaggle Notebook
BATCH_SIZE = 32
NUM_WORKERS = 0
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=False)
print('Stable DataLoaders created.')
print('Train batches:', len(train_loader))
print('Validation batches:', len(val_loader))
print('Test batches:', len(test_loader))


In [ ]:
# 59. Reload Best Baseline Checkpoint
baseline_best = OCTBaselineCNN(num_classes=len(CLASSES)).to(DEVICE)
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=False)
baseline_best.load_state_dict(checkpoint['model_state_dict'])
baseline_best.eval()
print('Loaded checkpoint from epoch:', checkpoint['epoch'])
print('Saved validation Macro F1:', checkpoint['val_macro_f1'])


In [ ]:
# 60. Best Baseline Validation Evaluation
baseline_val_loss, baseline_val_metrics, baseline_val_true, baseline_val_pred = evaluate_model(baseline_best, val_loader, criterion, DEVICE)
print('Best Baseline Validation Results')
print('-' * 40)
print(f'Loss:            {baseline_val_loss:.4f}')
print(f"Accuracy:        {baseline_val_metrics['accuracy']:.4f}")
print(f"Macro Precision: {baseline_val_metrics['macro_precision']:.4f}")
print(f"Macro Recall:    {baseline_val_metrics['macro_recall']:.4f}")
print(f"Macro F1:        {baseline_val_metrics['macro_f1']:.4f}")


In [ ]:
# 61. Per-Class Validation Metrics
from sklearn.metrics import classification_report
baseline_report = classification_report(baseline_val_true, baseline_val_pred, target_names=CLASSES, digits=4, zero_division=0)
print('Baseline CNN — Validation Classification Report\n')
print(baseline_report)


In [ ]:
# 64. Normalized Confusion Matrix
cm_normalized = confusion_matrix(baseline_val_true, baseline_val_pred, normalize='true')
display_cm_norm = ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=CLASSES)
fig, ax = plt.subplots(figsize=(8, 7))
display_cm_norm.plot(ax=ax, cmap='Blues', values_format='.2f', colorbar=False)
plt.title('Baseline CNN — Normalized Validation Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# 65. Balanced Accuracy
from sklearn.metrics import balanced_accuracy_score
baseline_balanced_accuracy = balanced_accuracy_score(baseline_val_true, baseline_val_pred)
print('Validation Accuracy:', round(baseline_val_metrics['accuracy'], 4))
print('Validation Balanced Accuracy:', round(baseline_balanced_accuracy, 4))
print('Validation Macro F1:', round(baseline_val_metrics['macro_f1'], 4))


### E01 Results

The compact CNN achieved:

- Validation Accuracy: **0.9490**
- Validation Balanced Accuracy: **0.9045**
- Validation Macro F1: **0.9141**

Performance was strong overall, but class-level analysis showed lower
recall for DME and DRUSEN than for CNV and NORMAL.

This motivated a controlled experiment examining whether class-weighted
training could improve minority-class sensitivity.

# 6. Experiment E02 — Class-Weighted Cross-Entropy

The training population is imbalanced, with substantially fewer DME and
DRUSEN images than CNV and NORMAL images.

Experiment E02 therefore keeps the CNN architecture, preprocessing,
optimizer, learning rate, and training duration unchanged while replacing
standard cross-entropy with inverse-frequency class-weighted
cross-entropy.

### Hypothesis

Increasing the contribution of minority-class samples may improve DME
and DRUSEN recall.

The possible trade-off is reduced precision caused by excessive
minority-class predictions.

In [ ]:
# 68. Calculate Class Weights
train_class_counts = train_final_df['class'].value_counts().reindex(CLASSES)
TOTAL_TRAIN = len(train_final_df)
NUM_CLASSES = len(CLASSES)
class_weights = []
for class_name in CLASSES:
    class_count = train_class_counts[class_name]
    weight = TOTAL_TRAIN / (NUM_CLASSES * class_count)
    class_weights.append(weight)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
weight_table = pd.DataFrame({'class': CLASSES, 'train_images': train_class_counts.values, 'class_weight': class_weights})
display(weight_table.round(4))


In [ ]:
# 69. Reset Random Seeds for E02
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
set_all_seeds(SEED)
print(f'Random seeds reset to {SEED}')


In [ ]:
# 70. Initialize E02 Model
weighted_model = OCTBaselineCNN(num_classes=len(CLASSES)).to(DEVICE)
weighted_total_params, weighted_trainable_params = count_parameters(weighted_model)
print(f'Total parameters: {weighted_total_params:,}')
print(f'Trainable parameters: {weighted_trainable_params:,}')


In [ ]:
# 71. E02 Loss and Optimizer
weighted_criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
weighted_optimizer = torch.optim.Adam(weighted_model.parameters(), lr=0.001)
print('Loss: Weighted CrossEntropyLoss')
print('Optimizer: Adam')
print('Learning rate: 0.001')
print('Class weights:', class_weights_tensor)


In [ ]:
# 72. E02 Training Configuration
E02_EPOCHS = 8
E02_MODEL_PATH = '/kaggle/working/oct_weighted_ce_best.pt'
e02_history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': [], 'train_macro_f1': [], 'val_macro_f1': []}
e02_best_val_f1 = -1.0
e02_best_epoch = 0


In [ ]:
# 73. Train E02 — Weighted Cross-Entropy
for epoch in range(1, E02_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{E02_EPOCHS}')
    print('-' * 50)
    train_loss, train_metrics = train_one_epoch(weighted_model, train_loader, weighted_optimizer, weighted_criterion, DEVICE)
    val_loss, val_metrics, val_true, val_pred = evaluate_model(weighted_model, val_loader, weighted_criterion, DEVICE)
    e02_history['train_loss'].append(train_loss)
    e02_history['val_loss'].append(val_loss)
    e02_history['train_accuracy'].append(train_metrics['accuracy'])
    e02_history['val_accuracy'].append(val_metrics['accuracy'])
    e02_history['train_macro_f1'].append(train_metrics['macro_f1'])
    e02_history['val_macro_f1'].append(val_metrics['macro_f1'])
    print(f'Train Loss: {train_loss:.4f}')
    print(f"Train Accuracy: {train_metrics['accuracy']:.4f}")
    print(f"Train Macro F1: {train_metrics['macro_f1']:.4f}")
    print(f'\nVal Loss: {val_loss:.4f}')
    print(f"Val Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"Val Macro F1: {val_metrics['macro_f1']:.4f}")
    if val_metrics['macro_f1'] > e02_best_val_f1:
        e02_best_val_f1 = val_metrics['macro_f1']
        e02_best_epoch = epoch
        torch.save({'epoch': epoch, 'model_state_dict': weighted_model.state_dict(), 'optimizer_state_dict': weighted_optimizer.state_dict(), 'val_macro_f1': e02_best_val_f1, 'class_weights': class_weights, 'class_to_idx': CLASS_TO_IDX, 'train_mean': TRAIN_MEAN, 'train_std': TRAIN_STD}, E02_MODEL_PATH)
        print('\n✓ Best E02 model saved.')
print('\nE02 training complete.')
print('Best epoch:', e02_best_epoch)
print('Best validation Macro F1:', round(e02_best_val_f1, 4))


In [ ]:
epochs_e02 = range(1, E02_EPOCHS + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs_e02, e02_history['train_macro_f1'], marker='o', label='Train')
plt.plot(epochs_e02, e02_history['val_macro_f1'], marker='o', label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Macro F1')
plt.title('E02 Weighted CE — Macro F1')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 74. Load Best E02 Model
e02_best_model = OCTBaselineCNN(num_classes=len(CLASSES)).to(DEVICE)
e02_checkpoint = torch.load(E02_MODEL_PATH, map_location=DEVICE, weights_only=False)
e02_best_model.load_state_dict(e02_checkpoint['model_state_dict'])
e02_best_model.eval()
print('Loaded E02 epoch:', e02_checkpoint['epoch'])
print('Saved validation Macro F1:', e02_checkpoint['val_macro_f1'])


In [ ]:
# 75. E02 Validation Evaluation
e02_val_loss, e02_val_metrics, e02_val_true, e02_val_pred = evaluate_model(e02_best_model, val_loader, weighted_criterion, DEVICE)
print(classification_report(e02_val_true, e02_val_pred, target_names=CLASSES, digits=4, zero_division=0))


In [ ]:
e02_cm_normalized = confusion_matrix(e02_val_true, e02_val_pred, normalize='true')
e02_display = ConfusionMatrixDisplay(confusion_matrix=e02_cm_normalized, display_labels=CLASSES)
fig, ax = plt.subplots(figsize=(8, 7))
e02_display.plot(ax=ax, cmap='Blues', values_format='.2f', colorbar=False)
plt.title('E02 Weighted CE — Normalized Validation Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
e02_balanced_accuracy = balanced_accuracy_score(e02_val_true, e02_val_pred)
print('Accuracy:', round(e02_val_metrics['accuracy'], 4))
print('Balanced Accuracy:', round(e02_balanced_accuracy, 4))
print('Macro F1:', round(e02_val_metrics['macro_f1'], 4))


In [ ]:
e02_history_df = pd.DataFrame({'Epoch': range(1, len(e02_history['train_loss']) + 1), 'Train Loss': e02_history['train_loss'], 'Val Loss': e02_history['val_loss'], 'Train Accuracy': e02_history['train_accuracy'], 'Val Accuracy': e02_history['val_accuracy'], 'Train Macro F1': e02_history['train_macro_f1'], 'Val Macro F1': e02_history['val_macro_f1']})
display(e02_history_df.round(4))


In [ ]:
E02_HISTORY_FILE = '/kaggle/working/e02_training_history.csv'
e02_history_df.to_csv(E02_HISTORY_FILE, index=False)
print('E02 training history saved to:')
print(E02_HISTORY_FILE)


### E02 Results

Class weighting increased minority-class recall:

- DME recall: **0.8323 → 0.8920**
- DRUSEN recall: **0.8403 → 0.9389**

However, the weighted model also produced substantially more false
positive minority-class predictions.

Overall Macro F1 decreased:

**0.9141 → 0.8891**

Balanced accuracy increased slightly, but precision deteriorated.

Therefore, full inverse-frequency weighting was not retained for later
experiments.

This result demonstrates that class-imbalance correction is not
automatically beneficial and should be assessed using multiple metrics.

# 7. Experiment E03 — Aspect-Preserving OCT Preprocessing

Dataset exploration showed substantial variation in OCT image width.

Directly stretching every image to 224 × 224 changes the original retinal
geometry.

Experiment E03 therefore evaluates whether preserving image aspect ratio
improves classification.

The CNN architecture and standard cross-entropy loss remain unchanged.

The only major modification is preprocessing:

Original OCT  
↓  
Aspect-Preserving Resize  
↓  
Symmetric Padding  
↓  
224 × 224 Input

In [ ]:
# 77. Aspect-Preserving Resize + Padding
from torchvision.transforms import functional as TF

class AspectPreservingResizePad:

    def __init__(self, target_size=224, fill=0):
        self.target_size = target_size
        self.fill = fill

    def __call__(self, image):
        width, height = image.size
        scale = self.target_size / max(width, height)
        new_width = int(round(width * scale))
        new_height = int(round(height * scale))
        image = TF.resize(image, [new_height, new_width], interpolation=InterpolationMode.BILINEAR, antialias=True)
        pad_width = self.target_size - new_width
        pad_height = self.target_size - new_height
        left = pad_width // 2
        right = pad_width - left
        top = pad_height // 2
        bottom = pad_height - top
        image = TF.pad(image, [left, top, right, bottom], fill=self.fill)
        return image


In [ ]:
# 78. Compare Preprocessing Strategies
example_row = train_final_df.sample(1, random_state=SEED).iloc[0]
example_path = example_row['path']
example_class = example_row['class']
with Image.open(example_path) as img:
    original_image = img.convert('L')
direct_resize = transforms.Resize((224, 224), interpolation=InterpolationMode.BILINEAR, antialias=True)(original_image)
aspect_processor = AspectPreservingResizePad(target_size=224, fill=0)
aspect_preserved = aspect_processor(original_image)
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
axes[0].imshow(original_image, cmap='gray')
axes[0].set_title(f'Original\n{original_image.size}')
axes[1].imshow(direct_resize, cmap='gray')
axes[1].set_title('E01 Direct Resize\n224 × 224')
axes[2].imshow(aspect_preserved, cmap='gray')
axes[2].set_title('E03 Aspect Preserved\n224 × 224')
for ax in axes:
    ax.axis('off')
plt.suptitle(f'Preprocessing Comparison — {example_class}')
plt.tight_layout()
plt.show()


In [ ]:
# 79. E03 Training Normalization Statistics
E03_STATS_SAMPLE_SIZE = 2000
e03_stats_sample = train_final_df.sample(n=min(E03_STATS_SAMPLE_SIZE, len(train_final_df)), random_state=SEED)
e03_stats_transform = transforms.Compose([AspectPreservingResizePad(target_size=224, fill=0), transforms.ToTensor()])
pixel_sum = 0.0
pixel_squared_sum = 0.0
pixel_count = 0
for image_path in e03_stats_sample['path']:
    with Image.open(image_path) as img:
        img = img.convert('L')
        tensor = e03_stats_transform(img)
    pixel_sum += tensor.sum().item()
    pixel_squared_sum += (tensor ** 2).sum().item()
    pixel_count += tensor.numel()
E03_MEAN = pixel_sum / pixel_count
E03_VARIANCE = pixel_squared_sum / pixel_count - E03_MEAN ** 2
E03_STD = E03_VARIANCE ** 0.5
print(f'E03 training mean: {E03_MEAN:.4f}')
print(f'E03 training std:  {E03_STD:.4f}')


In [ ]:
# 80. E03 Transforms
e03_train_transform = transforms.Compose([AspectPreservingResizePad(target_size=224, fill=0), transforms.RandomHorizontalFlip(p=0.5), transforms.ToTensor(), transforms.Normalize(mean=[E03_MEAN], std=[E03_STD])])
e03_eval_transform = transforms.Compose([AspectPreservingResizePad(target_size=224, fill=0), transforms.ToTensor(), transforms.Normalize(mean=[E03_MEAN], std=[E03_STD])])
print('E03 transforms created.')


In [ ]:
# 81. E03 Dataset Objects
e03_train_dataset = OCTDataset(train_final_df, transform=e03_train_transform)
e03_val_dataset = OCTDataset(val_final_df, transform=e03_eval_transform)
print('E03 training images:', len(e03_train_dataset))
print('E03 validation images:', len(e03_val_dataset))


In [ ]:
# 82. E03 DataLoaders
e03_train_loader = DataLoader(e03_train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
e03_val_loader = DataLoader(e03_val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
print('E03 training batches:', len(e03_train_loader))
print('E03 validation batches:', len(e03_val_loader))


In [ ]:
# 83. Initialize E03
set_all_seeds(SEED)
e03_model = OCTBaselineCNN(num_classes=len(CLASSES)).to(DEVICE)
e03_total_params, _ = count_parameters(e03_model)
print('E03 parameters:', f'{e03_total_params:,}')


In [ ]:
# 84. E03 Loss and Optimizer
e03_criterion = nn.CrossEntropyLoss()
e03_optimizer = torch.optim.Adam(e03_model.parameters(), lr=0.001)
print('E03 Loss: Standard CrossEntropy')
print('E03 Optimizer: Adam')
print('E03 Learning rate: 0.001')


In [ ]:
E03_EPOCHS = 8
E03_MODEL_PATH = '/kaggle/working/oct_e03_aspect_preserving_best.pt'
e03_history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': [], 'train_macro_f1': [], 'val_macro_f1': []}
e03_best_val_f1 = -1.0
e03_best_epoch = 0


In [ ]:
# 85. Train E03
for epoch in range(1, E03_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{E03_EPOCHS}')
    print('-' * 50)
    train_loss, train_metrics = train_one_epoch(e03_model, e03_train_loader, e03_optimizer, e03_criterion, DEVICE)
    val_loss, val_metrics, val_true, val_pred = evaluate_model(e03_model, e03_val_loader, e03_criterion, DEVICE)
    e03_history['train_loss'].append(train_loss)
    e03_history['val_loss'].append(val_loss)
    e03_history['train_accuracy'].append(train_metrics['accuracy'])
    e03_history['val_accuracy'].append(val_metrics['accuracy'])
    e03_history['train_macro_f1'].append(train_metrics['macro_f1'])
    e03_history['val_macro_f1'].append(val_metrics['macro_f1'])
    print(f'Train Loss: {train_loss:.4f}')
    print(f"Train Accuracy: {train_metrics['accuracy']:.4f}")
    print(f"Train Macro F1: {train_metrics['macro_f1']:.4f}")
    print(f'\nVal Loss: {val_loss:.4f}')
    print(f"Val Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"Val Macro F1: {val_metrics['macro_f1']:.4f}")
    if val_metrics['macro_f1'] > e03_best_val_f1:
        e03_best_val_f1 = val_metrics['macro_f1']
        e03_best_epoch = epoch
        torch.save({'epoch': epoch, 'model_state_dict': e03_model.state_dict(), 'optimizer_state_dict': e03_optimizer.state_dict(), 'val_macro_f1': e03_best_val_f1, 'train_mean': E03_MEAN, 'train_std': E03_STD, 'preprocessing': 'AspectPreservingResizePad', 'class_to_idx': CLASS_TO_IDX}, E03_MODEL_PATH)
        print('\n✓ Best E03 model saved.')
print('\nE03 training complete.')
print('Best epoch:', e03_best_epoch)
print('Best validation Macro F1:', round(e03_best_val_f1, 4))


In [ ]:
e03_history_df = pd.DataFrame({'Epoch': range(1, len(e03_history['train_loss']) + 1), 'Train Loss': e03_history['train_loss'], 'Val Loss': e03_history['val_loss'], 'Train Accuracy': e03_history['train_accuracy'], 'Val Accuracy': e03_history['val_accuracy'], 'Train Macro F1': e03_history['train_macro_f1'], 'Val Macro F1': e03_history['val_macro_f1']})
display(e03_history_df.round(4))
e03_history_df.to_csv('/kaggle/working/e03_training_history.csv', index=False)


In [ ]:
# 86. Load Best E03 Model
e03_best_model = OCTBaselineCNN(num_classes=len(CLASSES)).to(DEVICE)
e03_checkpoint = torch.load(E03_MODEL_PATH, map_location=DEVICE, weights_only=False)
e03_best_model.load_state_dict(e03_checkpoint['model_state_dict'])
e03_best_model.eval()
print('Loaded E03 checkpoint from epoch:', e03_checkpoint['epoch'])
print('Saved validation Macro F1:', e03_checkpoint['val_macro_f1'])


In [ ]:
# 87. E03 Validation Evaluation
e03_val_loss, e03_val_metrics, e03_val_true, e03_val_pred = evaluate_model(e03_best_model, e03_val_loader, e03_criterion, DEVICE)
print('E03 Validation Results')
print('-' * 40)
print(f'Loss:            {e03_val_loss:.4f}')
print(f"Accuracy:        {e03_val_metrics['accuracy']:.4f}")
print(f"Macro Precision: {e03_val_metrics['macro_precision']:.4f}")
print(f"Macro Recall:    {e03_val_metrics['macro_recall']:.4f}")
print(f"Macro F1:        {e03_val_metrics['macro_f1']:.4f}")


In [ ]:
# 88. E03 Per-Class Metrics
e03_report = classification_report(e03_val_true, e03_val_pred, target_names=CLASSES, digits=4, zero_division=0)
print('E03 — Validation Classification Report\n')
print(e03_report)


In [ ]:
# 89. E03 Balanced Accuracy
e03_balanced_accuracy = balanced_accuracy_score(e03_val_true, e03_val_pred)
print('Accuracy:', round(e03_val_metrics['accuracy'], 4))
print('Balanced Accuracy:', round(e03_balanced_accuracy, 4))
print('Macro F1:', round(e03_val_metrics['macro_f1'], 4))


In [ ]:
# 90. E03 Normalized Confusion Matrix
e03_cm_normalized = confusion_matrix(e03_val_true, e03_val_pred, normalize='true')
e03_display = ConfusionMatrixDisplay(confusion_matrix=e03_cm_normalized, display_labels=CLASSES)
fig, ax = plt.subplots(figsize=(8, 7))
e03_display.plot(ax=ax, cmap='Blues', values_format='.2f', colorbar=False)
plt.title('E03 Aspect-Preserving CNN — Normalized Validation Confusion Matrix')
plt.tight_layout()
plt.show()


### E03 Results

Aspect-preserving preprocessing improved validation performance without
increasing model size.

- Validation Accuracy: **0.9528**
- Validation Balanced Accuracy: **0.9195**
- Validation Macro F1: **0.9268**

Macro F1 improved compared with E01:

**0.9141 → 0.9268**

The improvement was particularly strong for DME.

The best checkpoint occurred at epoch 7. Validation performance declined
at epoch 8 despite continued improvement on the training set, indicating
the beginning of overfitting.

# 8. Experiment E04 — Transfer Learning with ResNet18

Experiment E04 investigates whether pretrained visual representations can
improve OCT classification.

The successful aspect-preserving preprocessing strategy from E03 is
retained.

Because ResNet18 is pretrained on RGB ImageNet images, each grayscale OCT
image is replicated across three channels and normalized using ImageNet
statistics.

### Model

- Architecture: pretrained ResNet18
- Parameters: **11,178,564**
- Loss: standard cross-entropy
- Optimizer: Adam
- Learning rate: 0.0001
- Model-selection metric: validation Macro F1

In [ ]:
# 92. Import ResNet18
from torchvision.models import resnet18, ResNet18_Weights
print('ResNet18 modules imported successfully.')


In [ ]:
# 93. E04 Preprocessing
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
e04_train_transform = transforms.Compose([AspectPreservingResizePad(target_size=224, fill=0), transforms.RandomHorizontalFlip(p=0.5), transforms.Grayscale(num_output_channels=3), transforms.ToTensor(), transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])
e04_eval_transform = transforms.Compose([AspectPreservingResizePad(target_size=224, fill=0), transforms.Grayscale(num_output_channels=3), transforms.ToTensor(), transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])
print('E04 transforms created.')


In [ ]:
# 94. E04 Dataset Objects
e04_train_dataset = OCTDataset(train_final_df, transform=e04_train_transform)
e04_val_dataset = OCTDataset(val_final_df, transform=e04_eval_transform)
print('E04 training images:', len(e04_train_dataset))
print('E04 validation images:', len(e04_val_dataset))


In [ ]:
# 95. E04 DataLoaders
E04_BATCH_SIZE = 32
e04_train_loader = DataLoader(e04_train_dataset, batch_size=E04_BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
e04_val_loader = DataLoader(e04_val_dataset, batch_size=E04_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
print('Training batches:', len(e04_train_loader))
print('Validation batches:', len(e04_val_loader))


In [ ]:
# 97. Initialize Pretrained ResNet18
set_all_seeds(SEED)
weights = ResNet18_Weights.DEFAULT
e04_model = resnet18(weights=weights)


In [ ]:
num_features = e04_model.fc.in_features
e04_model.fc = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(num_features, len(CLASSES)))
e04_model = e04_model.to(DEVICE)
print(e04_model.fc)


In [ ]:
# 98. E04 Parameter Count
e04_total_params, e04_trainable_params = count_parameters(e04_model)
print(f'Total parameters: {e04_total_params:,}')
print(f'Trainable parameters: {e04_trainable_params:,}')


In [ ]:
# 100. E04 Loss and Optimizer
e04_criterion = nn.CrossEntropyLoss()
e04_optimizer = torch.optim.Adam(e04_model.parameters(), lr=0.0001)
print('Loss: Standard CrossEntropy')
print('Optimizer: Adam')
print('Learning rate: 0.0001')


In [ ]:
# 101. E04 Training Configuration
E04_EPOCHS = 6
E04_MODEL_PATH = '/kaggle/working/oct_e04_resnet18_best.pt'
e04_history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': [], 'train_macro_f1': [], 'val_macro_f1': []}
e04_best_val_f1 = -1.0
e04_best_epoch = 0


In [ ]:
# 102. Train E04 ResNet18
for epoch in range(1, E04_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{E04_EPOCHS}')
    print('-' * 50)
    train_loss, train_metrics = train_one_epoch(e04_model, e04_train_loader, e04_optimizer, e04_criterion, DEVICE)
    val_loss, val_metrics, val_true, val_pred = evaluate_model(e04_model, e04_val_loader, e04_criterion, DEVICE)
    e04_history['train_loss'].append(train_loss)
    e04_history['val_loss'].append(val_loss)
    e04_history['train_accuracy'].append(train_metrics['accuracy'])
    e04_history['val_accuracy'].append(val_metrics['accuracy'])
    e04_history['train_macro_f1'].append(train_metrics['macro_f1'])
    e04_history['val_macro_f1'].append(val_metrics['macro_f1'])
    print(f'Train Loss: {train_loss:.4f}')
    print(f"Train Accuracy: {train_metrics['accuracy']:.4f}")
    print(f"Train Macro F1: {train_metrics['macro_f1']:.4f}")
    print(f'\nVal Loss: {val_loss:.4f}')
    print(f"Val Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"Val Macro F1: {val_metrics['macro_f1']:.4f}")
    if val_metrics['macro_f1'] > e04_best_val_f1:
        e04_best_val_f1 = val_metrics['macro_f1']
        e04_best_epoch = epoch
        torch.save({'epoch': epoch, 'model_state_dict': e04_model.state_dict(), 'optimizer_state_dict': e04_optimizer.state_dict(), 'val_macro_f1': e04_best_val_f1, 'architecture': 'ResNet18', 'pretrained': True, 'preprocessing': 'AspectPreservingResizePad', 'class_to_idx': CLASS_TO_IDX}, E04_MODEL_PATH)
        print('\n✓ Best E04 model saved.')
print('\nE04 training complete.')
print('Best epoch:', e04_best_epoch)
print('Best validation Macro F1:', round(e04_best_val_f1, 4))


In [ ]:
# 103. Save E04 History
e04_history_df = pd.DataFrame({'Epoch': range(1, len(e04_history['train_loss']) + 1), 'Train Loss': e04_history['train_loss'], 'Val Loss': e04_history['val_loss'], 'Train Accuracy': e04_history['train_accuracy'], 'Val Accuracy': e04_history['val_accuracy'], 'Train Macro F1': e04_history['train_macro_f1'], 'Val Macro F1': e04_history['val_macro_f1']})
display(e04_history_df.round(4))
e04_history_df.to_csv('/kaggle/working/e04_training_history.csv', index=False)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(e04_history_df['Epoch'], e04_history_df['Train Macro F1'], marker='o', label='Train')
plt.plot(e04_history_df['Epoch'], e04_history_df['Val Macro F1'], marker='o', label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Macro F1')
plt.title('E04 Pretrained ResNet18 — Macro F1')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 104. Load Best E04 ResNet18
e04_best_model = resnet18(weights=None)
num_features = e04_best_model.fc.in_features
e04_best_model.fc = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(num_features, len(CLASSES)))
e04_best_model = e04_best_model.to(DEVICE)
e04_checkpoint = torch.load(E04_MODEL_PATH, map_location=DEVICE, weights_only=False)
e04_best_model.load_state_dict(e04_checkpoint['model_state_dict'])
e04_best_model.eval()
print('Loaded E04 checkpoint from epoch:', e04_checkpoint['epoch'])
print('Saved validation Macro F1:', e04_checkpoint['val_macro_f1'])


In [ ]:
# 105. E04 Validation Evaluation
e04_val_loss, e04_val_metrics, e04_val_true, e04_val_pred = evaluate_model(e04_best_model, e04_val_loader, e04_criterion, DEVICE)
print('E04 Validation Results')
print('-' * 40)
print(f'Loss:            {e04_val_loss:.4f}')
print(f"Accuracy:        {e04_val_metrics['accuracy']:.4f}")
print(f"Macro Precision: {e04_val_metrics['macro_precision']:.4f}")
print(f"Macro Recall:    {e04_val_metrics['macro_recall']:.4f}")
print(f"Macro F1:        {e04_val_metrics['macro_f1']:.4f}")


In [ ]:
# 106. E04 Per-Class Validation Metrics
e04_report = classification_report(e04_val_true, e04_val_pred, target_names=CLASSES, digits=4, zero_division=0)
print('E04 ResNet18 — Validation Classification Report\n')
print(e04_report)


In [ ]:
# 107. E04 Balanced Accuracy
e04_balanced_accuracy = balanced_accuracy_score(e04_val_true, e04_val_pred)
print('Accuracy:', round(e04_val_metrics['accuracy'], 4))
print('Balanced Accuracy:', round(e04_balanced_accuracy, 4))
print('Macro F1:', round(e04_val_metrics['macro_f1'], 4))


In [ ]:
# 108. E04 Normalized Confusion Matrix
e04_cm_normalized = confusion_matrix(e04_val_true, e04_val_pred, normalize='true')
e04_display = ConfusionMatrixDisplay(confusion_matrix=e04_cm_normalized, display_labels=CLASSES)
fig, ax = plt.subplots(figsize=(8, 7))
e04_display.plot(ax=ax, cmap='Blues', values_format='.2f', colorbar=False)
plt.title('E04 ResNet18 — Normalized Validation Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# 109. E01–E04 Comparison
all_experiments_df = pd.DataFrame({'Experiment': ['E01 Custom CNN + Direct Resize', 'E02 Custom CNN + Weighted CE', 'E03 Custom CNN + Aspect Preserving', 'E04 Pretrained ResNet18'], 'Parameters': [total_params, weighted_total_params, e03_total_params, e04_total_params], 'Accuracy': [baseline_val_metrics['accuracy'], e02_val_metrics['accuracy'], e03_val_metrics['accuracy'], e04_val_metrics['accuracy']], 'Balanced Accuracy': [baseline_balanced_accuracy, e02_balanced_accuracy, e03_balanced_accuracy, e04_balanced_accuracy], 'Macro Precision': [baseline_val_metrics['macro_precision'], e02_val_metrics['macro_precision'], e03_val_metrics['macro_precision'], e04_val_metrics['macro_precision']], 'Macro Recall': [baseline_val_metrics['macro_recall'], e02_val_metrics['macro_recall'], e03_val_metrics['macro_recall'], e04_val_metrics['macro_recall']], 'Macro F1': [baseline_val_metrics['macro_f1'], e02_val_metrics['macro_f1'], e03_val_metrics['macro_f1'], e04_val_metrics['macro_f1']]})
display(all_experiments_df.round(4))


In [ ]:
all_experiments_df.to_csv('/kaggle/working/e01_to_e04_comparison.csv', index=False)


In [ ]:
# 110. E03 vs E04 Efficiency Trade-off
parameter_ratio = e04_total_params / e03_total_params
macro_f1_gain = e04_val_metrics['macro_f1'] - e03_val_metrics['macro_f1']
accuracy_gain = e04_val_metrics['accuracy'] - e03_val_metrics['accuracy']
e03_fp32_size_mb = e03_total_params * 4 / 1024 ** 2
e04_fp32_size_mb = e04_total_params * 4 / 1024 ** 2
print(f'ResNet18 has approximately {parameter_ratio:.1f}× more parameters.')
print(f'Macro F1 improvement: {macro_f1_gain:+.4f}')
print(f'Accuracy improvement: {accuracy_gain:+.4f}')
print(f'\nApproximate FP32 parameter memory:')
print(f'E03 Custom CNN: {e03_fp32_size_mb:.2f} MB')
print(f'E04 ResNet18:   {e04_fp32_size_mb:.2f} MB')


### E04 Results

ResNet18 achieved the strongest validation performance:

- Validation Accuracy: **0.9568**
- Validation Balanced Accuracy: **0.9386**
- Validation Macro F1: **0.9351**

Compared with the custom E03 CNN, ResNet18 improved Macro F1 by only
0.0083 on validation but used approximately **28.7× more parameters**.

At this stage, the compact CNN therefore appeared surprisingly competitive
with the much larger pretrained network.

The independent test evaluation later revealed an important difference in
generalization robustness.

# 9. Validation Experiment Comparison

| Experiment | Main Modification | Parameters | Accuracy | Balanced Accuracy | Macro F1 |
|---|---|---:|---:|---:|---:|
| E01 | Custom CNN + direct resize | 389,828 | 0.9490 | 0.9045 | 0.9141 |
| E02 | Custom CNN + weighted CE | 389,828 | 0.9244 | 0.9151 | 0.8891 |
| E03 | Custom CNN + aspect-preserving preprocessing | 389,828 | 0.9528 | 0.9195 | 0.9268 |
| **E04** | **Pretrained ResNet18** | **11,178,564** | **0.9568** | **0.9386** | **0.9351** |

The experiments show that:

- aggressive inverse-frequency weighting improved recall but harmed overall
  class-balanced performance;
- preserving image geometry improved the compact CNN without increasing its
  parameter count;
- ResNet18 achieved the strongest validation performance;
- validation performance alone was not sufficient to determine robustness.

# 10. Explainability with Grad-CAM

Classification metrics alone do not reveal which image regions influence
a model's decision.

Grad-CAM was therefore applied to the final ResNet18 model.

Representative correct predictions generally showed attention concentrated
around retinal-image structures rather than exclusively in the surrounding
padding.

High-confidence errors from different patients were also examined.

These cases demonstrate that the network may focus on abnormal retinal
regions even when the final disease classification is incorrect.

Grad-CAM should nevertheless be interpreted cautiously: it provides coarse
localization and does not prove that the network has learned clinically
validated pathological biomarkers.


In [ ]:
# 111. Collect E04 Validation Predictions
def collect_prediction_probabilities(model, loader, device):
    model.eval()
    all_true = []
    all_pred = []
    all_probs = []
    with torch.no_grad():
        for batch_images, batch_labels in tqdm(loader, desc='Collecting predictions', leave=False):
            batch_images = batch_images.to(device, non_blocking=True)
            with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                logits = model(batch_images)
            probabilities = torch.softmax(logits.float(), dim=1)
            predictions = probabilities.argmax(dim=1)
            all_true.extend(batch_labels.numpy())
            all_pred.extend(predictions.cpu().numpy())
            all_probs.append(probabilities.cpu().numpy())
    all_probs = np.concatenate(all_probs, axis=0)
    return (np.array(all_true), np.array(all_pred), all_probs)


In [ ]:
e04_true, e04_pred, e04_probs = collect_prediction_probabilities(e04_best_model, e04_val_loader, DEVICE)
e04_val_results = val_final_df.reset_index(drop=True).copy()
e04_val_results['dataset_index'] = np.arange(len(e04_val_results))
e04_val_results['true_idx'] = e04_true
e04_val_results['pred_idx'] = e04_pred
e04_val_results['predicted_class'] = [IDX_TO_CLASS[i] for i in e04_pred]
e04_val_results['confidence'] = e04_probs.max(axis=1)
e04_val_results['correct'] = e04_true == e04_pred
for idx, class_name in enumerate(CLASSES):
    e04_val_results[f'prob_{class_name}'] = e04_probs[:, idx]
print('Validation predictions:', len(e04_val_results))
print('Correct predictions:', e04_val_results['correct'].sum())
print('Errors:', (~e04_val_results['correct']).sum())
display(e04_val_results[['filename', 'class', 'predicted_class', 'confidence', 'correct']].head())


In [ ]:
e04_val_results.to_csv('/kaggle/working/e04_validation_predictions.csv', index=False)
print('Validation prediction table saved.')


In [ ]:
# 112. Confidence Analysis
confidence_summary = e04_val_results.groupby('correct')['confidence'].agg(['count', 'mean', 'median', 'min', 'max'])
display(confidence_summary.round(4))


In [ ]:
print('Mean confidence — correct predictions:')
print(round(e04_val_results.loc[e04_val_results['correct'], 'confidence'].mean(), 4))
print('\nMean confidence — incorrect predictions:')
print(round(e04_val_results.loc[~e04_val_results['correct'], 'confidence'].mean(), 4))


In [ ]:
# 113. Identify Cases for Explainability
high_confidence_errors = e04_val_results[~e04_val_results['correct']].sort_values('confidence', ascending=False).head(12)
uncertain_correct = e04_val_results[e04_val_results['correct']].sort_values('confidence', ascending=True).head(12)
print('Highest-confidence errors:')
display(high_confidence_errors[['filename', 'class', 'predicted_class', 'confidence']])
print('\nLowest-confidence correct predictions:')
display(uncertain_correct[['filename', 'class', 'predicted_class', 'confidence']])


In [ ]:
# 114. Error Pair Analysis
errors_only = e04_val_results[~e04_val_results['correct']]
error_pairs = errors_only.groupby(['class', 'predicted_class']).size().reset_index(name='count').sort_values('count', ascending=False)
display(error_pairs)


In [ ]:
# 115. Grad-CAM Implementation
import torch.nn.functional as F

class GradCAM:

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.forward_handle = target_layer.register_forward_hook(self._save_activations)
        self.backward_handle = target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self.activations = output

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, image_tensor, target_class=None):
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        logits = self.model(image_tensor)
        if target_class is None:
            target_class = logits.argmax(dim=1).item()
        score = logits[0, target_class]
        score.backward()
        gradients = self.gradients
        activations = self.activations
        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=image_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam_min = cam.min()
        cam_max = cam.max()
        cam = (cam - cam_min) / (cam_max - cam_min + 1e-08)
        return (cam[0, 0].detach().cpu().numpy(), logits.detach().cpu())

    def remove_hooks(self):
        self.forward_handle.remove()
        self.backward_handle.remove()


In [ ]:
gradcam = GradCAM(model=e04_best_model, target_layer=e04_best_model.layer4[-1].conv2)
print('Grad-CAM initialized.')


In [ ]:
# 116. Grad-CAM Visualization Helpers
def tensor_to_display_image(tensor):
    tensor = tensor.detach().cpu().clone()
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    tensor = tensor * std + mean
    tensor = tensor.clamp(0, 1)
    grayscale = tensor[0].numpy()
    return grayscale


In [ ]:
def create_gradcam_overlay(grayscale, heatmap, alpha=0.4):
    gray_rgb = np.repeat(grayscale[..., np.newaxis], 3, axis=2)
    heatmap_rgb = plt.get_cmap('jet')(heatmap)[..., :3]
    overlay = (1 - alpha) * gray_rgb + alpha * heatmap_rgb
    return np.clip(overlay, 0, 1)


In [ ]:
# 117. Plot Grad-CAM Examples
def show_gradcam_cases(cases_df, title, max_cases=6):
    cases_df = cases_df.head(max_cases).reset_index(drop=True)
    num_cases = len(cases_df)
    fig, axes = plt.subplots(num_cases, 3, figsize=(12, 4 * num_cases))
    if num_cases == 1:
        axes = np.expand_dims(axes, axis=0)
    for row_idx, row in cases_df.iterrows():
        dataset_index = int(row['dataset_index'])
        image_tensor, true_label = e04_val_dataset[dataset_index]
        input_tensor = image_tensor.unsqueeze(0).to(DEVICE)
        target_class = int(row['pred_idx'])
        heatmap, logits = gradcam.generate(input_tensor, target_class=target_class)
        display_image = tensor_to_display_image(image_tensor)
        overlay = create_gradcam_overlay(display_image, heatmap)
        axes[row_idx, 0].imshow(display_image, cmap='gray')
        axes[row_idx, 0].set_title(f"Original\nTrue: {row['class']}")
        axes[row_idx, 1].imshow(heatmap, cmap='jet')
        axes[row_idx, 1].set_title('Grad-CAM')
        axes[row_idx, 2].imshow(overlay)
        axes[row_idx, 2].set_title(f"Pred: {row['predicted_class']}\nConfidence: {row['confidence']:.3f}")
        for col in range(3):
            axes[row_idx, col].axis('off')
    plt.suptitle(title, fontsize=15)
    plt.tight_layout()
    plt.show()


In [ ]:
# 118. Correct Predictions — One per Class
correct_examples = []
for class_name in CLASSES:
    class_cases = e04_val_results[e04_val_results['correct'] & (e04_val_results['class'] == class_name)].sort_values('confidence', ascending=False).head(1)
    correct_examples.append(class_cases)
correct_examples_df = pd.concat(correct_examples, ignore_index=True)
display(correct_examples_df[['filename', 'class', 'predicted_class', 'confidence']])
show_gradcam_cases(correct_examples_df, title='E04 Grad-CAM — High-Confidence Correct Predictions', max_cases=4)


# 11. Patient-Level Failure Analysis

Image-level metrics can hide subject-level failure patterns.

Validation errors were therefore grouped by patient ID to determine whether
mistakes were broadly distributed or concentrated among a smaller number
of difficult subjects.


In [ ]:
# 120. Detailed Analysis of Patient 7675318
DIFFICULT_PATIENT = '7675318'
patient_7675318 = e04_val_results[e04_val_results['patient_id'] == DIFFICULT_PATIENT].copy()
print('Images from patient:', len(patient_7675318))
print('\nTrue class distribution:')
print(patient_7675318['class'].value_counts())
print('\nPredicted class distribution:')
print(patient_7675318['predicted_class'].value_counts())
print('\nAccuracy for this patient:')
print(round(patient_7675318['correct'].mean(), 4))
print('\nMean prediction confidence:')
print(round(patient_7675318['confidence'].mean(), 4))


In [ ]:
display(patient_7675318[['filename', 'class', 'predicted_class', 'confidence', 'correct']].sort_values('confidence', ascending=False))


In [ ]:
# 121. Patient-Level Error Analysis
patient_error_summary = e04_val_results.groupby('patient_id').agg(total_images=('correct', 'size'), correct_images=('correct', 'sum'), mean_confidence=('confidence', 'mean')).reset_index()
patient_error_summary['errors'] = patient_error_summary['total_images'] - patient_error_summary['correct_images']
patient_error_summary['error_rate'] = patient_error_summary['errors'] / patient_error_summary['total_images']
patient_error_summary = patient_error_summary.sort_values(['errors', 'error_rate'], ascending=False)
display(patient_error_summary.head(20))


In [ ]:
# 122. Error Concentration Across Patients
total_errors = (~e04_val_results['correct']).sum()
error_by_patient = e04_val_results[~e04_val_results['correct']].groupby('patient_id').size().sort_values(ascending=False)
print('Total validation errors:', total_errors)
print('Patients with at least one error:', len(error_by_patient))
for top_n in [1, 5, 10, 20]:
    errors_from_top = error_by_patient.head(top_n).sum()
    percentage = errors_from_top / total_errors * 100
    print(f'Top {top_n:2d} patients account for {percentage:.2f}% of all errors')


In [ ]:
# 123. High-Confidence Errors from Distinct Patients
distinct_patient_errors = e04_val_results[~e04_val_results['correct']].sort_values('confidence', ascending=False).drop_duplicates(subset='patient_id', keep='first').head(12)
display(distinct_patient_errors[['patient_id', 'filename', 'class', 'predicted_class', 'confidence']])


In [ ]:
show_gradcam_cases(distinct_patient_errors, title='E04 Grad-CAM — High-Confidence Errors from Distinct Patients', max_cases=6)


## Patient-Level Findings

Image-level errors were highly concentrated among particular patients.

Among **474 validation errors**:

- 95 patients contributed at least one error
- the most difficult patient accounted for **17.30%** of all errors
- the five most difficult patients accounted for **44.94%**
- the ten most difficult patients accounted for **56.12%**
- the twenty most difficult patients accounted for **68.78%**

One notable subject, patient `7675318`, contributed 67 validation images
across CNV, DME, and DRUSEN categories.

For this patient:

- Image-level accuracy: **0.6418**
- Mean prediction confidence: **0.9435**

Several DME scans from this subject were classified as CNV with extremely
high confidence.

This analysis demonstrates that aggregate image-level accuracy can hide
substantial subject-level variability.


# 12. Confidence and Calibration

Several individual predictions were confidently incorrect, so model
confidence was evaluated separately from classification accuracy.


In [ ]:
# 124. Confidence Distribution
correct_confidence = e04_val_results.loc[e04_val_results['correct'], 'confidence']
incorrect_confidence = e04_val_results.loc[~e04_val_results['correct'], 'confidence']
plt.figure(figsize=(8, 5))
plt.hist(correct_confidence, bins=30, alpha=0.6, label='Correct')
plt.hist(incorrect_confidence, bins=30, alpha=0.6, label='Incorrect')
plt.xlabel('Maximum Predicted Probability')
plt.ylabel('Number of Images')
plt.title('E04 Prediction Confidence')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 125. High-Confidence Error Rates
for threshold in [0.8, 0.9, 0.95, 0.99]:
    selected = e04_val_results[e04_val_results['confidence'] >= threshold]
    if len(selected) == 0:
        continue
    error_rate = (~selected['correct']).mean()
    print(f'Confidence >= {threshold:.2f}')
    print(f'  Samples: {len(selected)}')
    print(f'  Error rate: {error_rate:.4f}')


In [ ]:
# 126. Expected Calibration Error
def expected_calibration_error(confidences, correct, n_bins=15):
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    bin_results = []
    for i in range(n_bins):
        lower = bin_edges[i]
        upper = bin_edges[i + 1]
        if i == 0:
            mask = (confidences >= lower) & (confidences <= upper)
        else:
            mask = (confidences > lower) & (confidences <= upper)
        count = mask.sum()
        if count == 0:
            continue
        mean_confidence = confidences[mask].mean()
        accuracy = correct[mask].mean()
        fraction = count / len(confidences)
        ece += fraction * abs(accuracy - mean_confidence)
        bin_results.append({'lower': lower, 'upper': upper, 'count': count, 'accuracy': accuracy, 'confidence': mean_confidence})
    return (ece, pd.DataFrame(bin_results))


In [ ]:
e04_ece, calibration_df = expected_calibration_error(e04_val_results['confidence'].values, e04_val_results['correct'].values.astype(float), n_bins=15)
print('Expected Calibration Error:', round(e04_ece, 4))
display(calibration_df.round(4))


In [ ]:
# 127. Reliability Diagram
plt.figure(figsize=(7, 6))
plt.plot(calibration_df['confidence'], calibration_df['accuracy'], marker='o', label='ResNet18')
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect calibration')
plt.xlabel('Mean Confidence')
plt.ylabel('Observed Accuracy')
plt.title('E04 Reliability Diagram')
plt.legend()
plt.tight_layout()
plt.show()


## Calibration Findings

The ResNet18 model achieved an Expected Calibration Error of:

**ECE = 0.023**

Among predictions with confidence ≥ 0.99:

- **9,100 images** met the confidence threshold
- the observed error rate was approximately **0.63%**

The model was therefore reasonably calibrated overall, although a small
number of highly confident failures remained.

A low average calibration error does not eliminate the possibility of
individual severe overconfidence.


In [ ]:
# 128. Create Aspect-Preserving Content Mask
def create_content_mask(image_path, target_size=224):
    with Image.open(image_path) as img:
        width, height = img.size
    scale = target_size / max(width, height)
    new_width = int(round(width * scale))
    new_height = int(round(height * scale))
    left = (target_size - new_width) // 2
    top = (target_size - new_height) // 2
    mask = np.zeros((target_size, target_size), dtype=np.float32)
    mask[top:top + new_height, left:left + new_width] = 1.0
    return mask


In [ ]:
# 129. Grad-CAM Padding Sensitivity Analysis
CAM_ANALYSIS_SIZE = 200
cam_analysis_sample = e04_val_results.sample(n=min(CAM_ANALYSIS_SIZE, len(e04_val_results)), random_state=SEED)
cam_attention_records = []
for _, row in tqdm(cam_analysis_sample.iterrows(), total=len(cam_analysis_sample), desc='Grad-CAM region analysis'):
    dataset_index = int(row['dataset_index'])
    image_tensor, _ = e04_val_dataset[dataset_index]
    input_tensor = image_tensor.unsqueeze(0).to(DEVICE)
    heatmap, _ = gradcam.generate(input_tensor, target_class=int(row['pred_idx']))
    content_mask = create_content_mask(row['path'])
    total_cam = heatmap.sum() + 1e-08
    content_attention = (heatmap * content_mask).sum() / total_cam
    padding_attention = 1.0 - content_attention
    cam_attention_records.append({'correct': row['correct'], 'class': row['class'], 'predicted_class': row['predicted_class'], 'content_attention': content_attention, 'padding_attention': padding_attention})


In [ ]:
cam_attention_df = pd.DataFrame(cam_attention_records)
print('Mean Grad-CAM attention inside OCT image:')
print(round(cam_attention_df['content_attention'].mean(), 4))
print('\nMean Grad-CAM attention in padding:')
print(round(cam_attention_df['padding_attention'].mean(), 4))


In [ ]:
# 130. Grad-CAM Attention Enrichment
area_records = []
for _, row in cam_analysis_sample.iterrows():
    content_mask = create_content_mask(row['path'])
    content_area_fraction = content_mask.mean()
    area_records.append(content_area_fraction)
cam_attention_df['content_area_fraction'] = area_records
cam_attention_df['attention_enrichment'] = cam_attention_df['content_attention'] / cam_attention_df['content_area_fraction']
print('Mean fraction of canvas occupied by OCT image:', round(cam_attention_df['content_area_fraction'].mean(), 4))
print('Mean Grad-CAM attention inside OCT image:', round(cam_attention_df['content_attention'].mean(), 4))
print('Mean attention enrichment:', round(cam_attention_df['attention_enrichment'].mean(), 4))


## Padding Sensitivity

Because aspect-preserving preprocessing introduced artificial padded
regions, model attention was quantitatively evaluated with respect to the
original OCT image region.

Across the sampled validation images:

- Mean Grad-CAM attention inside OCT image content: **95.48%**
- Mean attention in padding: **4.52%**

This indicates that Grad-CAM activation was predominantly located within
the original OCT image region rather than the artificial padding.

Because the OCT content itself occupies a substantial portion of the
224 × 224 canvas, the attention-enrichment calculation above provides an
additional reference for interpreting this result. The numerical enrichment
value is intentionally left to the executed notebook output rather than
being hard-coded here.


In [ ]:
gradcam.remove_hooks()


# 13. Final Evaluation on the Untouched Test Set

All model-development decisions were finalized before examining the
official OCT2017 Version 3 test set.

E03 and E04 were then evaluated once on the 1,000-image official test
population.

No architecture, preprocessing, loss-function, hyperparameter, or
decision-threshold changes were made after examining these results.

In [ ]:
# 131. E03 Test Dataset
e03_test_dataset = OCTDataset(test_final_df, transform=e03_eval_transform)
e03_test_loader = DataLoader(e03_test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
print('E03 test images:', len(e03_test_dataset))
print('E03 test batches:', len(e03_test_loader))


In [ ]:
# 132. E04 Test Dataset
e04_test_dataset = OCTDataset(test_final_df, transform=e04_eval_transform)
e04_test_loader = DataLoader(e04_test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
print('E04 test images:', len(e04_test_dataset))
print('E04 test batches:', len(e04_test_loader))


In [ ]:
# 133. Final Test — E03
e03_test_loss, e03_test_metrics, e03_test_true, e03_test_pred = evaluate_model(e03_best_model, e03_test_loader, e03_criterion, DEVICE)
e03_test_balanced_accuracy = balanced_accuracy_score(e03_test_true, e03_test_pred)
print('E03 FINAL TEST RESULTS')
print('-' * 40)
print(f'Loss:              {e03_test_loss:.4f}')
print(f"Accuracy:          {e03_test_metrics['accuracy']:.4f}")
print(f'Balanced Accuracy: {e03_test_balanced_accuracy:.4f}')
print(f"Macro Precision:   {e03_test_metrics['macro_precision']:.4f}")
print(f"Macro Recall:      {e03_test_metrics['macro_recall']:.4f}")
print(f"Macro F1:          {e03_test_metrics['macro_f1']:.4f}")


In [ ]:
print(classification_report(e03_test_true, e03_test_pred, target_names=CLASSES, digits=4, zero_division=0))


In [ ]:
# 134. Final Test — E04 ResNet18
e04_test_loss, e04_test_metrics, e04_test_true, e04_test_pred = evaluate_model(e04_best_model, e04_test_loader, e04_criterion, DEVICE)
e04_test_balanced_accuracy = balanced_accuracy_score(e04_test_true, e04_test_pred)
print('E04 FINAL TEST RESULTS')
print('-' * 40)
print(f'Loss:              {e04_test_loss:.4f}')
print(f"Accuracy:          {e04_test_metrics['accuracy']:.4f}")
print(f'Balanced Accuracy: {e04_test_balanced_accuracy:.4f}')
print(f"Macro Precision:   {e04_test_metrics['macro_precision']:.4f}")
print(f"Macro Recall:      {e04_test_metrics['macro_recall']:.4f}")
print(f"Macro F1:          {e04_test_metrics['macro_f1']:.4f}")


In [ ]:
print(classification_report(e04_test_true, e04_test_pred, target_names=CLASSES, digits=4, zero_division=0))


In [ ]:
# 135. E04 Final Test Confusion Matrix
e04_test_cm = confusion_matrix(e04_test_true, e04_test_pred)
display_test_cm = ConfusionMatrixDisplay(confusion_matrix=e04_test_cm, display_labels=CLASSES)
fig, ax = plt.subplots(figsize=(8, 7))
display_test_cm.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
plt.title('E04 ResNet18 — Final Test Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# 136. Compact CNN vs ResNet18 — Final Test
final_test_comparison = pd.DataFrame({'Model': ['E03 Custom CNN', 'E04 ResNet18'], 'Parameters': [e03_total_params, e04_total_params], 'Approx FP32 Size MB': [e03_fp32_size_mb, e04_fp32_size_mb], 'Accuracy': [e03_test_metrics['accuracy'], e04_test_metrics['accuracy']], 'Balanced Accuracy': [e03_test_balanced_accuracy, e04_test_balanced_accuracy], 'Macro Precision': [e03_test_metrics['macro_precision'], e04_test_metrics['macro_precision']], 'Macro Recall': [e03_test_metrics['macro_recall'], e04_test_metrics['macro_recall']], 'Macro F1': [e03_test_metrics['macro_f1'], e04_test_metrics['macro_f1']]})
display(final_test_comparison.round(4))


In [ ]:
final_test_comparison.to_csv('/kaggle/working/final_test_comparison.csv', index=False)


In [ ]:
# 137. Validation-to-Test Generalization
generalization_df = pd.DataFrame({'Model': ['E03 Custom CNN', 'E04 ResNet18'], 'Validation Macro F1': [e03_val_metrics['macro_f1'], e04_val_metrics['macro_f1']], 'Test Macro F1': [e03_test_metrics['macro_f1'], e04_test_metrics['macro_f1']]})
generalization_df['Difference'] = generalization_df['Test Macro F1'] - generalization_df['Validation Macro F1']
display(generalization_df.round(4))


## Final Test Results

| Model | Parameters | Accuracy | Balanced Accuracy | Macro Precision | Macro Recall | Macro F1 |
|---|---:|---:|---:|---:|---:|---:|
| E03 Custom CNN | 389,828 | 0.7510 | 0.7510 | 0.8681 | 0.7510 | 0.7490 |
| **E04 ResNet18** | **11,178,564** | **0.9120** | **0.9120** | **0.9315** | **0.9120** | **0.9098** |

The independent test evaluation substantially changed the interpretation
of the validation experiments.

Although E03 and E04 appeared relatively close during validation, their
test performance differed considerably.

Validation Macro F1:

- E03: **0.9268**
- E04: **0.9351**

Test Macro F1:

- E03: **0.7490**
- E04: **0.9098**

The compact CNN therefore experienced a large validation-to-test
performance decline, while ResNet18 generalized much more robustly.

## Class-Level Test Performance

ResNet18 correctly classified:

- **248 / 250 CNV**
- **245 / 250 DME**
- **169 / 250 DRUSEN**
- **250 / 250 NORMAL**

DRUSEN was the principal remaining failure mode.

Among the 81 incorrectly classified DRUSEN images:

- **76 were predicted as CNV**
- **5 were predicted as NORMAL**

Thus, most remaining errors involved confusion between DRUSEN and CNV
rather than generalized failure across all four classes.

# 14. Validation–Test Distribution Shift

The large validation-to-test performance gap motivated a post-hoc
comparison of image geometry and intensity statistics between the
development and official test populations.


In [ ]:
# 138. Validation vs Test Image Geometry
def collect_image_geometry(dataframe):
    records = []
    for _, row in tqdm(dataframe.iterrows(), total=len(dataframe), leave=False):
        with Image.open(row['path']) as img:
            width, height = img.size
        records.append({'class': row['class'], 'width': width, 'height': height, 'aspect_ratio': width / height})
    return pd.DataFrame(records)


In [ ]:
val_geometry = collect_image_geometry(val_final_df)
test_geometry = collect_image_geometry(test_final_df)
print('VALIDATION GEOMETRY')
display(val_geometry.groupby('class')[['width', 'height', 'aspect_ratio']].mean().round(3))
print('\nTEST GEOMETRY')
display(test_geometry.groupby('class')[['width', 'height', 'aspect_ratio']].mean().round(3))


In [ ]:
# 139. Validation vs Test Intensity Statistics
def estimate_intensity_stats(dataframe, sample_size=None):
    if sample_size is not None:
        dataframe = dataframe.sample(n=min(sample_size, len(dataframe)), random_state=SEED)
    records = []
    for _, row in tqdm(dataframe.iterrows(), total=len(dataframe), leave=False):
        with Image.open(row['path']) as img:
            arr = np.asarray(img.convert('L'), dtype=np.float32) / 255.0
        records.append({'class': row['class'], 'mean_intensity': arr.mean(), 'std_intensity': arr.std()})
    return pd.DataFrame(records)


In [ ]:
val_intensity = estimate_intensity_stats(val_final_df, sample_size=2000)
test_intensity = estimate_intensity_stats(test_final_df)
print('VALIDATION')
display(val_intensity.groupby('class')[['mean_intensity', 'std_intensity']].mean().round(4))
print('\nTEST')
display(test_intensity.groupby('class')[['mean_intensity', 'std_intensity']].mean().round(4))


In [ ]:
# 140. E04 Test DRUSEN Error Analysis
test_results_e04 = test_final_df.reset_index(drop=True).copy()
test_results_e04['true_idx'] = e04_test_true
test_results_e04['pred_idx'] = e04_test_pred
test_results_e04['predicted_class'] = [IDX_TO_CLASS[i] for i in e04_test_pred]
test_results_e04['correct'] = e04_test_true == e04_test_pred
drusen_test = test_results_e04[test_results_e04['class'] == 'DRUSEN']
print('DRUSEN test images:', len(drusen_test))
print('\nPredicted labels:')
print(drusen_test['predicted_class'].value_counts())


In [ ]:
# 141. Test Patient-Level Error Distribution
test_results_e04['patient_id'] = test_final_df.reset_index(drop=True)['patient_id']
test_patient_summary = test_results_e04.groupby('patient_id').agg(total_images=('correct', 'size'), correct_images=('correct', 'sum')).reset_index()
test_patient_summary['errors'] = test_patient_summary['total_images'] - test_patient_summary['correct_images']
test_patient_summary['accuracy'] = test_patient_summary['correct_images'] / test_patient_summary['total_images']
display(test_patient_summary.sort_values('errors', ascending=False).head(20))


## Distribution-Shift Findings

A substantial shift was observed between the validation and official test
populations.

### Image Geometry

Mean validation aspect ratios ranged approximately from:

**1.08 to 1.46**

The official test classes had mean aspect ratios around:

**2.10**

The test images were also substantially wider on average.

### Intensity Statistics

CNV, DME, and DRUSEN test images showed higher mean intensity and greater
intensity variation than their validation counterparts.

These findings indicate that the independent test population differs
measurably from the development population.

The analysis does not prove that geometric or intensity differences alone
caused the reduction in performance. However, the results provide evidence
of a meaningful distribution shift.

The compact CNN was considerably more sensitive to this shift, whereas the
pretrained ResNet18 retained much stronger performance.


# 15. Key Findings

### Dataset integrity matters

The initial OCT2017 Version 2 partition contained subject overlap and an
extremely small validation set. Dataset auditing therefore changed the
entire experimental design before model training began.

### Patient-disjoint evaluation is essential

All development and evaluation partitions were verified to contain
non-overlapping patient IDs.

### Class weighting is not automatically beneficial

Inverse-frequency weighting improved DME and DRUSEN recall but increased
false positive minority-class predictions and reduced Macro F1.

### Preprocessing affects performance

Aspect-preserving preprocessing improved the compact CNN validation Macro
F1 from **0.9141 to 0.9268** without increasing model complexity.

### Validation performance can be misleading

The compact CNN achieved strong validation performance but dropped from
**0.9268 validation Macro F1 to 0.7490 test Macro F1**.

### Transfer learning improved robustness

ResNet18 achieved **0.9098 test Macro F1**, substantially outperforming
the compact CNN on the independent test population.

### DRUSEN remained the most difficult category

The dominant final-model error was DRUSEN being classified as CNV.

### Failure analysis adds information beyond accuracy

Prediction errors were concentrated among a relatively small number of
patients, demonstrating that image-level metrics alone do not fully
describe model behavior.

# 16. Limitations

This project has several limitations.

1. The experiments use a single public retinal OCT dataset and therefore
   do not establish generalization to other hospitals, scanners, clinical
   centers, or populations.

2. The official test set contains only 1,000 images, with 250 samples per
   class.

3. Although patient-disjoint evaluation was used, substantial geometric
   and intensity differences were observed between the development and
   test populations.

4. Grad-CAM provides coarse localization and should not be interpreted as
   evidence that the model identified clinically validated biomarkers.

5. Dataset labels were used as provided and were not independently reviewed
   by ophthalmologists in this project.

6. High model confidence does not imply clinical certainty.

7. This work is a research benchmark and is not intended for clinical
   diagnosis or deployment.

# 17. Conclusion

This project developed a retinal OCT classification workflow through a
progressive sequence of dataset auditing, controlled deep-learning
experiments, explainability analysis, and independent evaluation.

The most important result was not simply the highest classification
accuracy.

The experiments showed that strong validation performance can substantially
overestimate performance on an independent population.

The compact CNN achieved competitive validation performance but was highly
sensitive to the distribution shift present in the official test set.

The pretrained ResNet18 required approximately 28.7 times more parameters,
but its advantage became substantial under independent evaluation,
achieving:

- **Test Accuracy: 91.2%**
- **Balanced Accuracy: 91.2%**
- **Macro F1: 0.9098**

The project highlights an important principle in medical image analysis:

> Reliable experimental design, patient-independent evaluation, dataset
> auditing, and failure analysis are as important as neural-network
> architecture selection.

The resulting system should therefore be viewed as a research benchmark
for studying OCT classification rather than as a clinically validated
diagnostic model.

# 18. Reproducibility

The experiments were implemented using Python and PyTorch.

Key design choices were fixed before final test evaluation:

- random seed: **42**
- patient-disjoint development split
- Macro F1 used for model selection
- official OCT2017 Version 3 test population kept untouched until final
  evaluation
- best model checkpoints selected using validation performance
- experimental split metadata saved for reproducibility

The associated GitHub repository will contain:

- the final notebook,
- reusable dataset and model code,
- experiment tables,
- figures,
- split metadata,
- dependency information,
- and project documentation.